# Avance 2 — Proyecto Final
## Variabilidad en el Tratamiento Oncológico y sus Efectos sobre la Mortalidad y la Estadía Hospitalaria en el Sistema Público Chileno

**Equipo:** Vicente · José Tomás · Sebastián  
**Dataset:** GRD Público MINSAL/FONASA 2019–2024  
**Fecha:** Abril 2026

---

### Contexto y Relevancia

El cáncer constituye la segunda causa de muerte en Chile y representa una carga creciente para el sistema de salud pública. Los GRD permiten estandarizar la clasificación de egresos hospitalarios, facilitando comparaciones entre establecimientos. Sin embargo, persisten brechas significativas en la gestión clínica: la duración de la estadía, la intensidad de procedimientos y los resultados en mortalidad varían considerablemente entre hospitales, incluso para pacientes con diagnósticos clínicamente comparables.

Este proyecto aborda esa variabilidad desde una perspectiva oncológica exclusiva, centrándose en el **cáncer gástrico (CIE-10 C16.\*)** como caso de estudio focal. Comprender si el hospital de atención actúa como determinante independiente de la trayectoria clínica tiene implicaciones directas para políticas de regionalización, asignación de recursos y garantías de equidad.

---

### Pregunta de Investigación

> **¿En qué medida el hospital de atención determina los días de estadía, la cantidad de procedimientos y la mortalidad intrahospitalaria en pacientes oncológicos clínicamente comparables (CIE-10 C16.\* — Cáncer Gástrico) del sistema público chileno?**

---

### Hipótesis

| Hipótesis | Enunciado | Test |
|---|---|---|
| **H₁** | La distribución de la cantidad de procedimientos difiere significativamente entre hospitales para pacientes con cáncer gástrico. | Kruskal-Wallis + Dunn-Bonferroni |
| **H₂** | La cantidad de procedimientos se asocia con la probabilidad de mortalidad intrahospitalaria, controlando por edad, severidad GRD y hospital. | Regresión Logística (OR, IC95%) |
| **H₃** | La cantidad de procedimientos predice la duración de la estadía hospitalaria, controlando por edad, severidad GRD y hospital. | Regresión OLS Múltiple (β, IC95%) |

Nivel de significancia: **α = 0.05**.

---
# 2. Descripción del Dataset

### Fuente
Los datos corresponden a los **GRD Públicos del Ministerio de Salud de Chile (MINSAL)**, gestionados por FONASA, correspondientes a los años **2019–2024**.

### Unidad de Análisis
Cada fila representa un **egreso hospitalario** (no un paciente único).

### Dimensiones
El dataset limpio (`GRD_Limpio.csv`) contiene aproximadamente **454.000 registros** y **145 variables**, de las cuales se seleccionan las clínicamente relevantes.

### Glosario de Variables Clave

| Variable | Tipo | Descripción |
|---|---|---|
| `diagnostico_principal` | Categórica (string) | Diagnóstico principal CIE-10 (ej: C16.9). |
| `dias_estada` | Numérica (int) | Días entre fecha de ingreso y fecha de alta. |
| `cantidad_procedimientos` | Numérica (int) | Conteo de procedimientos realizados. |
| `edad` | Numérica (float) | Edad del paciente en años al egreso. |
| `sexo` | Categórica (string) | Sexo biológico (MUJER / HOMBRE). |
| `mortalidad` | Binaria (bool) | `True` si falleció durante la hospitalización. |
| `severidad_grd` | Numérica (int) | Nivel de severidad GRD (1–4). |
| `peso_grd` | Numérica (float) | Peso relativo del GRD (proxy de consumo de recursos). |
| `hospital` | Categórica (int) | Código del establecimiento (`COD_HOSPITAL`). |
| `hospital_nombre` | Categórica (string) | Nombre del establecimiento de salud (mapeado desde `HospitalesGRD.xlsx`). |
| `tipo_ingreso` | Categórica (string) | Motivo de ingreso (URGENCIA, PROGRAMADA, OBSTETRICA). |
| `tipo_alta` | Categórica (string) | Condición de egreso (FALLECIDO, DOMICILIO, etc.). |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  2.1 Carga inicial del dataset                               ║
# ╚══════════════════════════════════════════════════════════════╝
from pathlib import Path
import pandas as pd
import numpy as np
import unicodedata
import re

DATA_PATH = Path('DATASET INICIAL') / 'GRD_Limpio.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'No se encontró el dataset en {DATA_PATH.resolve()}')

cols_uso = [
    'COD_HOSPITAL', 'hospital', 'diagnostico_principal', 'dias_estada',
    'edad', 'sexo', 'mortalidad', 'severidad_grd', 'peso_grd',
    'cantidad_procedimientos', 'TIPO_INGRESO', 'TIPOALTA',
    'comuna', 'region',
    'DIAGNOSTICO2', 'DIAGNOSTICO3', 'DIAGNOSTICO4', 'DIAGNOSTICO5', 'DIAGNOSTICO6', 'DIAGNOSTICO7', 'DIAGNOSTICO8', 'DIAGNOSTICO9', 'DIAGNOSTICO10', 'DIAGNOSTICO11', 'DIAGNOSTICO12', 'DIAGNOSTICO13', 'DIAGNOSTICO14', 'DIAGNOSTICO15', 'DIAGNOSTICO16', 'DIAGNOSTICO17', 'DIAGNOSTICO18', 'DIAGNOSTICO19', 'DIAGNOSTICO20', 'DIAGNOSTICO21', 'DIAGNOSTICO22', 'DIAGNOSTICO23', 'DIAGNOSTICO24', 'DIAGNOSTICO25', 'DIAGNOSTICO26', 'DIAGNOSTICO27', 'DIAGNOSTICO28', 'DIAGNOSTICO29', 'DIAGNOSTICO30', 'DIAGNOSTICO31', 'DIAGNOSTICO32', 'DIAGNOSTICO33', 'DIAGNOSTICO34', 'DIAGNOSTICO35'
]

df_raw = pd.read_csv(DATA_PATH, usecols=cols_uso, low_memory=False)

# ── Mapeo de códigos de hospital a nombres (para tablas; gráficos usan códigos) ──
HOSP_PATH = Path('DATASET INICIAL') / 'HospitalesGRD.xlsx'
if HOSP_PATH.exists():
    hosp_excel = pd.read_excel(HOSP_PATH)
    hosp_excel.columns = ['codigo', 'nombre']
    hosp_excel['codigo'] = hosp_excel['codigo'].astype(int)
    mapa_hospital = dict(zip(hosp_excel['codigo'], hosp_excel['nombre']))
    # Conservar código numérico en 'hospital' (usado en gráficos y análisis)
    # Crear columna aparte con el nombre completo para tablas/CSV
    df_raw['hospital_nombre'] = df_raw['hospital'].map(lambda x: mapa_hospital.get(int(x), str(int(x))) if pd.notna(x) else x)
    print(f'Mapeo de hospitales cargado: {len(mapa_hospital):,} establecimientos.')
    n_mapeados = df_raw['hospital_nombre'].notna().sum()
    print(f'Registros con nombre de hospital: {n_mapeados:,}')
else:
    df_raw['hospital_nombre'] = df_raw['hospital'].astype(str)
    print(f'Advertencia: No se encontró {HOSP_PATH}. Se usan códigos numéricos.')

print(f'Dataset cargado: {DATA_PATH}')
print(f'Observaciones totales: {len(df_raw):,}')
print(f'Variables cargadas: {df_raw.shape[1]}')
print()
print('Primeras filas:')
display(df_raw.head())

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  2.2 Resumen de tipos y valores faltantes                    ║
# ╚══════════════════════════════════════════════════════════════╝
resumen = pd.DataFrame({
    'Tipo': df_raw.dtypes,
    'No_Nulos': df_raw.count(),
    'Nulos': df_raw.isnull().sum(),
    'Unicos': df_raw.nunique()
})
display(resumen)
print(f'\nValores faltantes totales: {df_raw.isnull().sum().sum():,}')

---
# 3. Limpieza y Preparación de Datos

Los problemas identificados y sus soluciones:

1. **Registros no oncológicos:** Se filtran exclusivamente códigos CIE-10 C00–D49.
2. **Outliers extremos en estadía:** Corte en el percentil 99 de `dias_estada` para evitar sesgo por cuidados paliativos prolongados o errores de registro.
3. **Valores negativos/nulos:** Se eliminan registros con `dias_estada` inválidos.
4. **Conversión de tipos:** `mortalidad` (bool) → entero (0/1) para modelos logísticos.
5. **Homogeneización:** `sexo`, `TIPO_INGRESO` y `TIPOALTA` se normalizan a mayúsculas sin tildes.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  3.1 Filtrado oncológico (CIE-10: C00–D49)                   ║
# ╚══════════════════════════════════════════════════════════════╝
CODIGOS_PATH = Path('codigos_C00_D49.txt')

if CODIGOS_PATH.exists():
    with open(CODIGOS_PATH, encoding='utf-8', errors='ignore') as f:
        codigos_onco = {line.strip().upper() for line in f if line.strip()}
    print(f'Códigos oncológicos cargados desde archivo: {len(codigos_onco)}')
else:
    import itertools
    codigos_onco = set()
    for pref in ['C'] + [f'C{i:02d}' for i in range(1, 100)] + ['D'] + [f'D{i:02d}' for i in range(0, 50)]:
        codigos_onco.add(pref)
    print(f'Códigos generados programáticamente: {len(codigos_onco)}')

n_antes = len(df_raw)
df = df_raw[df_raw['diagnostico_principal'].isin(codigos_onco)].copy()
n_despues = len(df)

print(f'Registros antes del filtro: {n_antes:,}')
print(f'Registros después (C00–D49): {n_despues:,}')
print(f'Eliminados: {n_antes - n_despues:,} ({(n_antes - n_despues)/n_antes:.2%})')

### 3.2 Exclusión de Egresos Obstétricos

Se eliminan todos los registros con `tipo_ingreso == 'OBSTETRICA'`. Los egresos obstétricos carecen de relevancia clínica para el objetivo del proyecto (estudio de variabilidad oncológica) y exhiben patrones de estadía y procedimientos sistemáticamente distintos de los oncológicos. Su inclusión introduciría ruido en los modelos y distorsionaría las comparaciones inter-hospital al mezclar lógicas de atención clínicamente incomparables.

In [ ]:
# ── Excluir egresos obstétricos ──────────────────────────────────────────────
n_antes = len(df)
df = df[df['TIPO_INGRESO'].str.upper() != 'OBSTETRICA'].copy()
n_despues = len(df)
n_excluidos = n_antes - n_despues
print(f"Registros antes del filtro obstétrico : {n_antes:>10,}")
print(f"Registros excluidos (OBSTETRICA)       : {n_excluidos:>10,} ({n_excluidos/n_antes:.2%})")
print(f"Registros tras el filtro               : {n_despues:>10,}")

### 3.3 Creación de Variable: Comorbilidad

Se deriva la variable `comorbilidad` contabilizando los diagnósticos secundarios activos (DIAG2…DIAG30) que sean distintos del diagnóstico principal. Esta operacionalización captura la carga de enfermedad concomitante sin requerir índices externos como el Charlson-Deyo. Un mayor índice de comorbilidad se asocia, en la literatura clínica, con mayor complejidad de la hospitalización y mayor probabilidad de prolongación de estadía. La variable fue incorporada a solicitud del profesor para mejorar el control clínico de los modelos.

In [ ]:
# ── Derivar variable comorbilidad ────────────────────────────────────────────
# Identificar columnas de diagnósticos secundarios (DIAG2 en adelante)
diag_cols = [c for c in df.columns if c.upper().startswith('DIAG')
             and c.upper() not in ('DIAG1', 'DIAGNOSTICO1', 'CODIGO_DIAG1')
             and c != 'diagnostico_principal']

print(f"Columnas de diagnósticos secundarios encontradas: {len(diag_cols)}")
print(diag_cols[:5], "...")

def contar_comorbilidades(row):
    diag_ppal = str(row['diagnostico_principal']).strip().upper()
    count = 0
    for col in diag_cols:
        val = str(row[col]).strip().upper()
        if val not in ('', 'NAN', 'NONE', 'NAT') and val != diag_ppal:
            count += 1
    return count

df['comorbilidad'] = df.apply(contar_comorbilidades, axis=1)

print("\n=== Estadística descriptiva: comorbilidad ===")
print(df['comorbilidad'].describe().round(2))
print(f"\nPacientes sin comorbilidades (0): {(df['comorbilidad'] == 0).mean():.1%}")
print(f"Pacientes con 3+ comorbilidades : {(df['comorbilidad'] >= 3).mean():.1%}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  3.2 Limpieza de outliers y tipos                            ║
# ╚══════════════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt
import seaborn as sns

Path('outputs/graficos').mkdir(parents=True, exist_ok=True)
Path('outputs/tablas').mkdir(parents=True, exist_ok=True)
Path('outputs/inferencial').mkdir(parents=True, exist_ok=True)

df['dias_estada'] = pd.to_numeric(df['dias_estada'], errors='coerce')
df = df[df['dias_estada'].notna() & (df['dias_estada'] >= 0)].copy()

P_OUTLIER = df['dias_estada'].quantile(0.99)
df = df[df['dias_estada'] <= P_OUTLIER].copy()

df['mortalidad_int'] = df['mortalidad'].astype(int)

for col in ['sexo', 'TIPO_INGRESO', 'TIPOALTA']:
    df[col] = df[col].astype(str).str.strip().str.upper()

for col in ['edad', 'severidad_grd', 'peso_grd', 'cantidad_procedimientos']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

n_final = len(df)
print(f'Percentil 99 de días de estadía: {P_OUTLIER:.0f}')
print(f'Registros finales tras limpieza: {n_final:,}')
print()
print('Resumen post-limpieza (Oncología C00–D49):')
display(df[['dias_estada','edad','cantidad_procedimientos','severidad_grd','peso_grd','mortalidad_int']].describe().T)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  3.3 Subconjunto focal: Cáncer Gástrico (C16.*)              ║
# ╚══════════════════════════════════════════════════════════════╝
df_focus = df[df['diagnostico_principal'].str.startswith('C16', na=False)].copy()

print(f'Registros C16.* (Cáncer Gástrico): {len(df_focus):,}')
print(f'Proporción del total oncológico: {len(df_focus)/len(df):.2%}')
print()
print('Distribución de subcódigos C16:')
display(df_focus['diagnostico_principal'].value_counts().sort_index())

---
# 4. Análisis Exploratorio de Datos (EDA)

El EDA se estructura en dos niveles:
- **Nivel global (C00–D49):** Caracterización del universo oncológico para contextualizar magnitudes.
- **Nivel focal (C16.*):** Análisis bivariado controlado por diagnóstico, base inferencial del proyecto.

## 4.1 Estadística Descriptiva Global (Oncología C00–D49)

La Tabla 1 caracteriza el comportamiento clínico del conjunto de pacientes oncológicos (CIE-10: C00–D49) del sistema público chileno entre 2019 y 2024. Se destacan los siguientes hallazgos:

**Estadía hospitalaria:** La distribución de días de estadía presenta marcada asimetría positiva (mediana: 3 días; media: 6.1 días). Esta brecha entre mediana y media es característica de la oncología hospitalaria: la mayoría de los ingresos corresponde a procedimientos relativamente cortos —ciclos de quimioterapia, evaluaciones, biopsias ambulatorias—, mientras una cola de casos complejos (cirugías mayores, complicaciones postoperatorias, cuidados paliativos) eleva la media hacia arriba. El corte en el percentil 99 (44 días) elimina estadías extraordinarias sin afectar la estructura central de la distribución.

**Edad:** La mediana de 58.8 años es consistente con la epidemiología de las neoplasias en Chile, donde la mayor incidencia se concentra entre los 50 y 75 años. La desviación estándar de 20.9 años refleja la heterogeneidad del grupo oncológico, que abarca desde neoplasias pediátricas hasta tumores propios de la tercera edad. Al analizar el subgrupo de cáncer gástrico (C16.*) en la sección 4.2, la distribución etaria tenderá a desplazarse hacia edades mayores, consistente con el perfil epidemiológico de esta patología.

**Procedimientos:** Con una mediana de 7 procedimientos por hospitalización y una media de 8.2, los pacientes oncológicos reciben una carga diagnóstica y terapéutica considerable. La desviación estándar de 6.4 anticipa que el análisis inferencial encontrará diferencias significativas entre hospitales: distintos establecimientos gestionan de forma muy diferente la intensidad de la intervención para diagnósticos clínicamente similares.

**Mortalidad:** La tasa global de mortalidad intrahospitalaria del 5.22% es elevada en comparación con la población hospitalaria general, lo cual refleja la gravedad del grupo oncológico. Este indicador será central en el análisis bivariado (sección 4.2) y en las comparaciones entre grupos de neoplasias en Avance 3. Para el cáncer gástrico en particular, se espera una tasa de mortalidad intrahospitalaria superior al promedio oncológico, dado que un número importante de ingresos corresponde a estadios avanzados o a complicaciones agudas (obstrucción, hemorragia, perforación).

**Distribución por sexo:** La mayor proporción de mujeres (61.4%) refleja el peso que tienen las neoplasias mamarias y ginecológicas en el total oncológico del sistema público. Esta distribución varía significativamente al segmentar por tipo de cáncer: para el cáncer gástrico (C16.*), la proporción masculina es históricamente mayor, consistente con la epidemiología nacional e internacional de esta patología.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  4.1.1 Tabla 1 — Descriptivas globales                       ║
# ╚══════════════════════════════════════════════════════════════╝
cols_num = ['dias_estada', 'edad', 'cantidad_procedimientos', 'peso_grd', 'severidad_grd']
resumen = df[cols_num].describe().T
resumen['mediana'] = df[cols_num].median()
resumen['asimetria'] = df[cols_num].skew()
resumen['missing'] = df[cols_num].isnull().sum()
resumen = resumen[['count','mean','std','min','25%','mediana','75%','max','asimetria','missing']].round(2)
display(resumen)
resumen.to_csv('outputs/tablas/tabla1_descriptivas_global.csv')
print('Tabla 1 guardada en outputs/tablas/tabla1_descriptivas_global.csv')

### 4.1.3 Matriz de Correlación — Variables Numéricas (C00–D49)

La matriz de correlación de Pearson entre las variables numéricas del universo oncológico permite identificar asociaciones lineales y detectar posibles problemas de multicolinealidad antes de ajustar los modelos de regresión. Correlaciones |r| > 0.7 entre predictores indicarían multicolinealidad severa que podría inflar los errores estándar. La correlación entre la variable dependiente de interés (`dias_estada`) y las covariables orienta la selección de variables para el modelo OLS.

In [ ]:
# ── Matriz de correlación — variables numéricas oncológicas ─────────────────
vars_corr = ['dias_estada', 'edad', 'cantidad_procedimientos',
             'severidad_grd', 'peso_grd', 'mortalidad_int']
etiquetas_corr = {
    'dias_estada':              'Días de estadía',
    'edad':                     'Edad',
    'cantidad_procedimientos':  'N° Procedimientos',
    'severidad_grd':            'Severidad GRD',
    'peso_grd':                 'Peso GRD',
    'mortalidad_int':           'Mortalidad (0/1)',
}

corr_df = df[vars_corr].dropna()
corr_matrix = corr_df.corr(method='pearson')
tick_labels = [etiquetas_corr[v] for v in vars_corr]

# ── Gráfico de la matriz ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7.5))
fig.patch.set_facecolor('#FAFAFA')

mask_upper = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
cmap = sns.diverging_palette(220, 10, as_cmap=True)
hm = sns.heatmap(corr_matrix, mask=mask_upper, cmap=cmap,
                  vmin=-1, vmax=1, center=0,
                  annot=True, fmt='.2f', linewidths=0.5, linecolor='#EEEEEE',
                  annot_kws={'size': 11, 'weight': 'bold'},
                  cbar_kws={'label': 'Coeficiente de correlación de Pearson (r)', 'shrink': 0.8},
                  ax=ax)

ax.set_xticklabels(tick_labels, rotation=35, ha='right', fontsize=10.5)
ax.set_yticklabels(tick_labels, rotation=0, fontsize=10.5)
ax.set_title('Figura 3 — Matriz de Correlación de Pearson\nVariables Numéricas Clave · Universo Oncológico (C00–D49)',
             fontsize=13, fontweight='bold', pad=14)

ax.text(0.01, -0.14,
        '🔴 Rojo intenso: correlación negativa fuerte  |  🔵 Azul intenso: correlación positiva fuerte\n'
        'Triángulo inferior mostrado. Las correlaciones |r| > 0.70 merecen atención por multicolinealidad.',
        transform=ax.transAxes, fontsize=9, color='#555555')

plt.tight_layout()
plt.savefig('outputs/graficos/correlacion_variables_numericas.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla de correlaciones con p-valores
from scipy.stats import pearsonr
print("\n=== Correlaciones con dias_estada (Pearson r, p-valor) ===")
for v in vars_corr:
    if v != 'dias_estada':
        mask = corr_df[['dias_estada', v]].notna().all(axis=1)
        r, p = pearsonr(corr_df.loc[mask, 'dias_estada'], corr_df.loc[mask, v])
        stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
        print(f"  {etiquetas_corr[v]:25s}  r = {r:+.4f}  p = {p:.3e} {stars}")

### 4.1.4 Distribución de Variables Categóricas Clave (C00–D49)

Las estadísticas descriptivas de variables categóricas son complementarias a las numéricas: permiten entender la composición del universo oncológico en términos de sexo, tipo de ingreso, tipo de alta y distribución regional. Estas variables tienen relevancia directa para la pregunta de investigación: el tipo de ingreso es un predictor de mortalidad; la distribución por sexo condiciona el perfil epidemiológico del análisis focal.

In [ ]:
# ── Tablas de frecuencia — variables categóricas ─────────────────────────────
vars_cat = {
    'sexo':         'Sexo biológico',
    'TIPO_INGRESO': 'Tipo de ingreso',
    'TIPOALTA':     'Tipo de alta',
}

for col, nombre in vars_cat.items():
    freq = df[col].value_counts(dropna=False)
    pct  = (freq / len(df) * 100).round(2)
    tabla_cat = pd.DataFrame({'N': freq, '%': pct}).reset_index()
    tabla_cat.columns = [nombre, 'N', '%']
    print(f'\n── {nombre} (n total = {len(df):,}) ──')
    display(tabla_cat)

# ── Gráfico comparativo por sexo y tipo de ingreso ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#FAFAFA')

paleta = ['#2E75B6', '#C0392B', '#148F77', '#7F8C8D', '#F39C12']
for ax in axes:
    ax.set_facecolor('#FAFAFA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

for idx_ax, (col, nombre) in enumerate(vars_cat.items()):
    ax = axes[idx_ax]
    freq = df[col].value_counts(dropna=False).head(6)
    bars = ax.barh(freq.index.astype(str), freq.values,
                   color=paleta[:len(freq)], alpha=0.85, edgecolor='white')
    for bar, n, p in zip(bars, freq.values, (freq / len(df) * 100)):
        ax.text(bar.get_width() + freq.max() * 0.01, bar.get_y() + bar.get_height()/2,
                f'{n:,} ({p:.1f}%)', va='center', fontsize=9)
    ax.set_title(f'Figura 4{chr(65 + idx_ax)} — {nombre}', fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel('Frecuencia', fontsize=10)
    ax.set_xlim([0, freq.max() * 1.35])

plt.suptitle('Distribución de Variables Categóricas Clave — Universo Oncológico (C00–D49)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/graficos/categoricas_globales.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  4.1.2 Distribuciones univariadas                            ║
# ╚══════════════════════════════════════════════════════════════╝
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.histplot(df['dias_estada'], bins=50, kde=True, color='steelblue', ax=axes[0,0])
axes[0,0].set_title('Distribución de Días de Estadía (C00–D49)')
axes[0,0].set_xlabel('Días'); axes[0,0].set_ylabel('Frecuencia')

sns.histplot(df['edad'].dropna(), bins=50, kde=True, color='coral', ax=axes[0,1])
axes[0,1].set_title('Distribución de Edad (C00–D49)')
axes[0,1].set_xlabel('Años'); axes[0,1].set_ylabel('Frecuencia')

sns.histplot(df['cantidad_procedimientos'], bins=40, kde=False, color='seagreen', ax=axes[1,0])
axes[1,0].set_title('Distribución de Cantidad de Procedimientos (C00–D49)')
axes[1,0].set_xlabel('N° Procedimientos'); axes[1,0].set_ylabel('Frecuencia')

sns.countplot(x=df['severidad_grd'].dropna().astype(int), palette='viridis', ax=axes[1,1])
axes[1,1].set_title('Distribución de Severidad GRD (C00–D49)')
axes[1,1].set_xlabel('Severidad'); axes[1,1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig('outputs/graficos/eda_univariado_global.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 15 diagnósticos oncológicos
fig, ax = plt.subplots(figsize=(10, 6))
top_diag = df['diagnostico_principal'].value_counts().head(15)
sns.barplot(x=top_diag.values, y=top_diag.index, palette='magma', ax=ax)
ax.set_title('Top 15 Diagnósticos Oncológicos (CIE-10) — C00-D49')
ax.set_xlabel('Frecuencia')
plt.tight_layout()
plt.savefig('outputs/graficos/top15_diagnosticos_onco.png', dpi=150, bbox_inches='tight')
plt.show()

> **Interpretación de Figuras 1–2:** La distribución de días de estadía exhibe una marcada asimetría positiva (asimetría ≈ 3.5; mediana = 3 días; media ≈ 6 días), con una cola derecha extensa que refleja un subgrupo de pacientes con hospitalizaciones prolongadas por cirugías mayores, complicaciones postoperatorias o tratamiento paliativo intensivo. Esta asimetría es clínicamente coherente: la mayoría de los egresos oncológicos corresponden a procedimientos de corta estadía —ciclos de quimioterapia, biopsias, evaluaciones ambulatorias—, mientras un subgrupo de alta complejidad genera estadías superiores a 15 días que elevan la media de forma desproporcionada. La diferencia entre media y mediana justifica metodológicamente la transformación logarítmica utilizada como variable dependiente en el modelo OLS (Sección 5.5), ya que normaliza la distribución residual y estabiliza la varianza, cumpliendo los supuestos del modelo lineal. El cáncer gástrico (C16.*) se posiciona en el top de frecuencias del ranking diagnóstico, confirmando su representatividad para el análisis focal y su relevancia epidemiológica en el sistema público chileno.

## 4.2 EDA Bivariado — Controlado por Diagnóstico CIE-10 (C16.* — Cáncer Gástrico)

Para garantizar **comparabilidad clínica**, se seleccionan exclusivamente pacientes con diagnóstico principal C16.*, controlando así el efecto de heterogeneidad diagnóstica que invalidaría las comparaciones inter-hospital. Este grupo comprende desde C16.0 (cardias) hasta C16.9 (estómago SAI), e incluye todos los subtipos anatomopatológicos del cáncer gástrico registrados bajo CIE-10. La elección de C16.* como caso focal responde a tres criterios concurrentes: (1) alta prevalencia en el sistema público chileno, que asegura poder estadístico suficiente para el análisis por hospital; (2) elevada mortalidad intrahospitalaria relativa, que hace relevante la comparación de desenlaces entre establecimientos; y (3) heterogeneidad terapéutica conocida —cirugía curativa, quimioterapia perioperatoria, manejo paliativo—, que genera variabilidad natural en estadía y procedimientos que el análisis busca explicar. El EDA bivariado de esta sección constituye la base descriptiva que motiva las hipótesis de la Sección 5.

In [ ]:
# ── Parámetros para EDA bivariado ─────────────────────────────────────────
MIN_CASOS_HOSPITAL = 20

conteo_hosp = df_focus['hospital'].value_counts()
hospitales_validos = conteo_hosp[conteo_hosp >= MIN_CASOS_HOSPITAL].index
df_eda = df_focus[df_focus['hospital'].isin(hospitales_validos)].copy()

print(f'Hospitales incluidos (C16.*, ≥{MIN_CASOS_HOSPITAL} casos): {len(hospitales_validos)}')
print(f'Registros en subconjunto EDA: {len(df_eda):,}')

In [ ]:
# ── BOXPLOT: Días de estadía × Hospital (C16.*) ───────────────────────────
orden = df_eda.groupby('hospital')['dias_estada'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(16, 6))
sns.boxplot(x='hospital', y='dias_estada', data=df_eda, order=orden,
            palette='Spectral', showfliers=False, ax=ax)
ax.set_title('Distribución de Días de Estadía por Hospital\n(Diagnóstico controlado: C16.* — Cáncer Gástrico)')
ax.set_xlabel('Código Hospital'); ax.set_ylabel('Días de Estadía')
ax.tick_params(axis='x', rotation=45)
for label in ax.get_xticklabels():
    label.set_ha('right')
    label.set_fontsize(8)
plt.tight_layout()
plt.savefig('outputs/graficos/boxplot_dias_hospital_C16.png', dpi=150, bbox_inches='tight')
plt.show()

> **Interpretación del Boxplot — Días de Estadía por Hospital (C16.*):** Los hospitales del extremo izquierdo (mayores medianas) muestran cajas más anchas, indicando mayor dispersión interna para el mismo diagnóstico, lo que revela heterogeneidad en el manejo clínico independientemente de la severidad del caso. Esta variabilidad es clínicamente relevante: dentro del mismo código CIE-10, la diferencia entre el hospital con menor mediana (~3 días) y el de mayor mediana (~10 días) equivale a una semana adicional de hospitalización, con implicancias directas en costos para el sistema y en calidad de vida del paciente. La presencia de IQR reducidos en algunos centros refleja estandarización de protocolos (p. ej., vías clínicas para gastrectomía laparoscópica), mientras que la amplitud de otros sugiere ausencia de criterios uniformes de alta o mezcla de casos curativos y paliativos en el mismo establecimiento. Esta heterogeneidad entre hospitales es la evidencia descriptiva central que justifica el uso de efectos fijos por hospital en los modelos de regresión de la Sección 5, controlando así el efecto del establecimiento sobre los desenlaces. Es llamativo que hospitales de similar nivel de complejidad muestren diferencias de varianza considerables, lo que sugiere que la dispersión interna depende más de la organización hospitalaria que de la severidad del case-mix atendido.

In [ ]:
# ── BARPLOT: Tasa de mortalidad intrahospitalaria × Hospital (C16.*) ─────
mort_hosp = df_eda.groupby('hospital')['mortalidad_int'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(x=mort_hosp.index.astype(str), y=mort_hosp.values*100, palette='Reds_r', ax=ax)
ax.set_title('Tasa de Mortalidad Intrahospitalaria por Hospital (C16.*)')
ax.set_xlabel('Código Hospital'); ax.set_ylabel('Mortalidad (%)')
ax.tick_params(axis='x', rotation=45)
for label in ax.get_xticklabels():
    label.set_ha('right')
    label.set_fontsize(8)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig('outputs/graficos/barplot_mortalidad_hospital_C16.png', dpi=150, bbox_inches='tight')
plt.show()

> **Interpretación del Barplot — Tasa de Mortalidad por Hospital (C16.*):** La mortalidad intrahospitalaria varía entre aproximadamente 2% y 10% entre hospitales para el mismo diagnóstico C16.*, una brecha de cinco veces que resulta estadísticamente robusta dado el tamaño muestral de cada estrato. Esta diferencia no es explicable únicamente por la gravedad del cáncer, ya que el análisis controla por diagnóstico principal CIE-10 a cuatro dígitos, eliminando en gran parte la variación por subtipo tumoral y focalizando la comparación en factores organizacionales e institucionales del hospital. En términos clínicos, una tasa de mortalidad de 10% en un hospital de alta complejidad puede reflejar tanto un mayor volumen de casos avanzados derivados (función de referencia regional) como dificultades en el manejo postoperatorio o acceso tardío a unidades de cuidados intensivos. La heterogeneidad en mortalidad es la variable dependiente principal de la Hipótesis 2 (H₂) y constituye la motivación central para el modelo logístico con efectos fijos de hospital desarrollado en la Sección 5.4. Es notable que los hospitales con mayor mortalidad no coincidan necesariamente con los de mayor varianza en estadía, sugiriendo que los mecanismos que generan ambos fenómenos son parcialmente independientes.

In [ ]:
# ── BARPLOT: Promedio de procedimientos × Hospital (C16.*) ────────────────
proc_hosp = df_eda.groupby('hospital')['cantidad_procedimientos'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(x=proc_hosp.index.astype(str), y=proc_hosp.values, palette='Blues_r', ax=ax)
ax.set_title('Promedio de Procedimientos por Hospital (C16.*)')
ax.set_xlabel('Código Hospital'); ax.set_ylabel('Procedimientos promedio')
ax.tick_params(axis='x', rotation=45)
for label in ax.get_xticklabels():
    label.set_ha('right')
    label.set_fontsize(8)
for p in ax.patches:
    ax.annotate(f'{p.get_height():.1f}',
                (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig('outputs/graficos/barplot_procedimientos_hospital_C16.png', dpi=150, bbox_inches='tight')
plt.show()

> **Interpretación del Barplot — Promedio de Procedimientos por Hospital (C16.*):** La intensidad procedimental varía sustancialmente entre establecimientos: el rango observable entre el hospital con mayor promedio y el de menor promedio supera los 5 procedimientos por egreso, lo que representa una diferencia clínicamente significativa en términos de intervención diagnóstica y terapéutica. Hospitales con mayores promedios probablemente concentran cirugías oncológicas mayores (gastrectomías totales o subtotales con disección ganglionar D2), endoscopías de estadificación y procedimientos relacionados con quimioterapia perioperatoria, mientras que hospitales con menores promedios podrían estar gestionando casos predominantemente paliativos donde la intervención se limita al control de síntomas. Esta heterogeneidad procedimental es el núcleo de la Hipótesis 1 (H₁), que evalúa si la distribución de procedimientos difiere significativamente entre hospitales mediante el test de Kruskal-Wallis (Sección 5.3). La dirección del efecto es ambigua: más procedimientos puede reflejar mayor inversión terapéutica en casos curables, pero también mayor acumulación de intervenciones en pacientes que se deterioran, lo que plantea preguntas sobre la eficiencia clínica que serán retomadas en el análisis inferencial. Sorprendentemente, algunos hospitales de alta complejidad reconocida muestran promedios bajos, lo que podría indicar mayor eficiencia en la concentración quirúrgica o uso de protocolos que minimizan procedimientos innecesarios.

In [ ]:
# ── Tabla descriptiva comparativa por hospital (C16.*) ────────────────────
tabla_hosp = df_eda.groupby('hospital').agg(
    n=('dias_estada','size'),
    dias_media=('dias_estada','mean'),
    dias_mediana=('dias_estada','median'),
    dias_std=('dias_estada','std'),
    proc_media=('cantidad_procedimientos','mean'),
    proc_mediana=('cantidad_procedimientos','median'),
    mortalidad_tasa=('mortalidad_int','mean'),
    edad_media=('edad','mean'),
    severidad_moda=('severidad_grd', lambda x: x.mode()[0] if not x.mode().empty else np.nan)
).round(2).sort_values('dias_media', ascending=False)

# Agregar nombre completo del hospital para referencia
mapa_nombre = df_eda.drop_duplicates('hospital').set_index('hospital')['hospital_nombre'].to_dict()
tabla_hosp.insert(0, 'hospital_nombre', tabla_hosp.index.map(mapa_nombre))

display(tabla_hosp)
tabla_hosp.to_csv('outputs/tablas/tabla_descriptiva_hospital_C16.csv')
print('Tabla guardada en outputs/tablas/tabla_descriptiva_hospital_C16.csv')

### Análisis de Tabla 2 — Variación por Hospital

#### Variación de Estadía
Al comparar la dispersión interna de los días de estadía entre hospitales, se busca detectar qué tan consistente es el manejo clínico dentro de cada establecimiento para el mismo diagnóstico CIE-10 (C16.* — Cáncer Gástrico). Los hospitales con valores más altos de **dias_std** presentan mayor heterogeneidad clínica-operativa: dentro del mismo diagnóstico, algunos pacientes son dados de alta en pocos días mientras otros permanecen hospitalizados por períodos prolongados. Esto puede reflejar diferencias en la complejidad no capturada por el peso GRD (p. ej., estadio tumoral, necesidad de cirugía versus manejo médico), en los protocolos de alta, o en la disponibilidad de camas para continuar el tratamiento de forma ambulatoria. En contraste, los hospitales con menor varianza exhiben mayor estandarización en el manejo del cáncer gástrico, lo que es un indicador de procesos clínicos más protocolizados.

#### Mortalidad y Cantidad de Procedimientos
El cáncer gástrico (C16.*) se caracteriza por una mortalidad intrahospitalaria no despreciable, a diferencia de diagnósticos oncológicos de manejo principalmente ambulatorio. Un número importante de ingresos corresponde a estadios avanzados o a complicaciones agudas (obstrucción, hemorragia digestiva alta, perforación) que requieren intervención de urgencia. La relación entre **proc_media** y **mortalidad_tasa** debe interpretarse con cautela: una mayor intensidad de procedimientos puede reflejar tanto una intervención oportuna (factor protector) como la acumulación de procedimientos en pacientes que se deterioran en hospitales de menor resolución (indicador de mayor complejidad no capturada). Los hospitales de alta complejidad, que concentran derivaciones de casos graves, tenderán a mostrar mayor mortalidad sin que esto necesariamente refleje peor calidad asistencial.

#### Comparación del Peso GRD Medio entre Hospitales
El **peso_grd_media** permite verificar que las diferencias observadas entre hospitales no se deban a diferencias en la severidad basal de los pacientes. Para el cáncer gástrico (C16.*), el peso GRD esperado es mayor que en diagnósticos de baja complejidad, dado que estos casos frecuentemente implican cirugía mayor (gastrectomía total o subtotal), quimioterapia perioperatoria y manejo de complicaciones. Si el peso GRD medio es similar entre hospitales, la evidencia respalda que las diferencias en estadía y mortalidad se asocien a factores organizacionales o de práctica clínica local más que a cambios en la complejidad promedio de los casos. Si, por el contrario, el peso GRD varía significativamente entre establecimientos, debe incorporarse como variable de control en el análisis inferencial.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Funciones auxiliares para Tabla 3                           ║
# ╚══════════════════════════════════════════════════════════════╝
def normalizar_texto(txt):
    if txt is None or pd.isna(txt):
        return None
    txt = str(txt).strip().lower()
    txt = ''.join(
        c for c in unicodedata.normalize('NFKD', txt) if not unicodedata.combining(c)
    )
    txt = re.sub(r'\s+', ' ', txt)
    return txt

def moda_serie(s):
    s_valid = s.dropna()
    if s_valid.empty:
        return pd.NA
    return s_valid.mode().iloc[0]

print('Funciones auxiliares definidas.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  4.2.3 Tabla 3 — Varianza de Días de Estadía por Hospital   ║
# ╚══════════════════════════════════════════════════════════════╝
# ── Tabla de varianza por hospital + contexto socioeconómico (IDH) ─────────
def normalizar_llave_geo(valor):
    if valor is None or pd.isna(valor):
        return None
    return normalizar_texto(str(valor).replace('*', '').strip())

tabla_var = (
    df_eda.groupby('hospital')
    .agg(
        comuna=('comuna', moda_serie),
        region=('region', moda_serie),
        n=('dias_estada', 'count'),
        media=('dias_estada', 'mean'),
        mediana=('dias_estada', 'median'),
        std=('dias_estada', 'std'),
        varianza=('dias_estada', 'var'),
        p25=('dias_estada', lambda x: x.quantile(0.25)),
        p75=('dias_estada', lambda x: x.quantile(0.75)),
    )
    .reset_index()
)
tabla_var['iqr'] = tabla_var['p75'] - tabla_var['p25']
tabla_var['cv'] = tabla_var['std'] / tabla_var['media']
tabla_var['comuna_norm'] = tabla_var['comuna'].map(normalizar_llave_geo)
tabla_var['region_norm'] = tabla_var['region'].map(normalizar_llave_geo)

idh_path = DATA_PATH.parent / 'idh_comunas_2024.csv'
if idh_path.exists():
    idh = pd.read_csv(idh_path, sep='|', encoding='utf-8').rename(
        columns={
            'Region': 'region_idh',
            'Comuna': 'comuna_idh',
            'Tramo-IDH': 'tramo_idh_2024',
            'IDH': 'idh_comunal_2024',
        }
    )
    idh['comuna_norm'] = idh['comuna_idh'].map(normalizar_llave_geo)
    idh['region_norm'] = idh['region_idh'].map(normalizar_llave_geo)

    # Cruce principal por comuna y region.
    idh_cols = ['comuna_norm', 'region_norm', 'tramo_idh_2024', 'idh_comunal_2024']
    tabla_var = tabla_var.merge(idh[idh_cols], on=['comuna_norm', 'region_norm'], how='left')

    # Fallback por comuna cuando el nombre es unico en el archivo IDH.
    comunas_unicas = idh['comuna_norm'].value_counts()
    comunas_unicas = comunas_unicas[comunas_unicas == 1].index
    idh_unico = (
        idh[idh['comuna_norm'].isin(comunas_unicas)]
        .drop_duplicates('comuna_norm')
        .set_index('comuna_norm')
    )
    mask_fallback = tabla_var['idh_comunal_2024'].isna() & tabla_var['comuna_norm'].isin(comunas_unicas)
    tabla_var.loc[mask_fallback, 'idh_comunal_2024'] = (
        tabla_var.loc[mask_fallback, 'comuna_norm'].map(idh_unico['idh_comunal_2024'])
    )
    tabla_var.loc[mask_fallback, 'tramo_idh_2024'] = (
        tabla_var.loc[mask_fallback, 'comuna_norm'].map(idh_unico['tramo_idh_2024'])
    )

    cobertura_idh = tabla_var['idh_comunal_2024'].notna().mean() * 100
    print(f'IDH comunal incorporado en Tabla 3 (cobertura: {cobertura_idh:.1f}%).')
else:
    tabla_var['tramo_idh_2024'] = pd.NA
    tabla_var['idh_comunal_2024'] = np.nan
    print(f'Advertencia: no se encontro {idh_path}. Tabla 3 se exporta sin IDH.')

tabla_var = tabla_var.sort_values('varianza', ascending=False)
tabla_var.insert(0, 'rank', range(1, len(tabla_var) + 1))
tabla_var['idh_comunal_2024'] = pd.to_numeric(tabla_var['idh_comunal_2024'], errors='coerce')
tabla_var = tabla_var.drop(columns=['comuna_norm', 'region_norm']).round(3)

# Agregar nombre completo del hospital para referencia
mapa_nombre_var = df_eda.drop_duplicates('hospital').set_index('hospital')['hospital_nombre'].to_dict()
tabla_var.insert(1, 'hospital_nombre', tabla_var['hospital'].map(mapa_nombre_var))

display(tabla_var.head(20))
tabla_var.to_csv('outputs/tablas/03_varianza_por_hospital.csv', index=False)
print('Tabla 3 guardada: outputs/tablas/03_varianza_por_hospital.csv')

## Análisis de Tabla 3
#### Varianza de Días de Estadía por Hospital (mayor → menor) + Contexto IDH Comunal

---

La Tabla 3 confirma empíricamente que el establecimiento de atención actúa como un determinante independiente de la trayectoria clínica de los pacientes con cáncer gástrico (C16.*) en el sistema público chileno. Los resultados del test de Kruskal-Wallis (ver Sección 5.3) rechazan H₀ con alta significación estadística y un tamaño de efecto que supera el umbral de efecto grande definido por Cohen (η² > .14), indicando que el hospital de atención explica una proporción sustancial de la variabilidad en días de estadía, incluso al controlar por diagnóstico CIE-10 y peso GRD.

**Coherencia con la Literatura**

Estos hallazgos son plenamente consistentes con la evidencia internacional disponible. La variabilidad observada respalda los postulados fundacionales de Wennberg y Gittelsohn sobre las variaciones no justificadas en la práctica médica. En el ámbito del cáncer gástrico, los resultados concuerdan con Munir et al. (2024) y Kamaraju et al. (2022), quienes demostraron que los factores organizacionales alteran tanto la mortalidad como la duración de la estadía, incluso al aislar la severidad del paciente. Para esta patología en particular, la disponibilidad de cirugía oncológica especializada (gastrectomía con disección ganglionar D2) y de oncología médica para quimioterapia perioperatoria (esquemas FLOT o FOLFOX) son determinantes clave de los resultados, recursos que no están homogéneamente distribuidos en el sistema público chileno.

**Hospitales con Mayor Varianza Interna**

Los hospitales encabezando la Tabla 3 (mayor varianza en días de estadía) concentran la mayor heterogeneidad en el manejo del cáncer gástrico. Destacan, por ejemplo, Victoria (varianza 57.55; IDH 0.577), Copiapó (51.76; IDH 0.688), Ovalle (47.76; IDH 0.597), San Antonio (47.10; IDH 0.602) y Villarrica (47.03; IDH 0.563). Que un mismo establecimiento presente alta varianza interna dentro de un mismo diagnóstico CIE-10 puede atribuirse a: (1) diferencias en estadio tumoral no capturadas por GRD, (2) variabilidad en tiempos de resolución quirúrgica o inicio de quimioterapia, y (3) función de referencia regional para casos de mayor complejidad.

Desde una perspectiva territorial, una hipótesis plausible es que hospitales con alta varianza mezclen pacientes locales y derivados interregionales con perfiles clínicos muy distintos, ampliando la dispersión intrahospitalaria aun bajo un mismo código diagnóstico.

**Impacto Socioeconómico en la Varianza de Días de Estadía (IDH Comunal) — Evidencia de Tabla 3**

La incorporación del IDH comunal permite evaluar si existe un gradiente socioeconómico lineal entre contexto territorial y varianza. Con 59 hospitales con IDH disponible, la asociación lineal es prácticamente nula (Pearson = 0.048; Spearman = 0.042). Esto indica que no hay una relación monotónica simple del tipo "a menor IDH, mayor varianza" para todo el sistema.

Sin embargo, la ausencia de correlación global no implica ausencia de efecto socioeconómico. Al desagregar por tramos de IDH, aparecen patrones de heterogeneidad estructural:

- Tramo medio-bajo: mayor varianza promedio (36.87) y también mayor varianza ponderada por tamaño muestral (37.75).
- Tramo bajo: varianza promedio alta (34.47), cercana a tramos de mayor desarrollo.
- Tramo medio-alto: varianza ponderada menor (34.29) pese a concentrar la mayor cantidad de observaciones.
- Tramo alto: varianza ponderada intermedia (34.74), consistente con mezcla de hospitales de referencia y hospitales con procesos más protocolizados.

La comparación por extremos refuerza esta lectura no lineal: el cuartil de mayor varianza y el de menor varianza tienen IDH promedio casi idéntico (0.620 vs 0.622). Por tanto, la dispersión en estadía no depende solo del nivel de desarrollo comunal, sino de una interacción entre contexto socioeconómico, organización hospitalaria y posición en la red de derivación.

En términos de política pública, este resultado es importante: el IDH no opera como causa única, pero sí como modulador de riesgo operacional. En comunas de menor desarrollo, barreras de acceso, diagnóstico más tardío y menor continuidad ambulatoria pueden aumentar la inestabilidad del flujo clínico. En comunas con mayor IDH, la varianza puede mantenerse elevada cuando el hospital cumple rol de alta complejidad y absorbe case-mix derivado. La "lotería del hospital" observada en la Tabla 3 es, entonces, institucional y territorial al mismo tiempo.

**Posibles Sesgos y Limitaciones**

A pesar de la robustez del análisis estadístico, el diseño metodológico presenta posibles sesgos inherentes al uso de datos administrativos retrospectivos. El potencial sesgo de codificación es especialmente relevante en cáncer gástrico: la precisión de la clasificación CIE-10 (C16.0 a C16.9) depende de la calidad del registro médico y puede variar entre establecimientos, afectando la homogeneidad del grupo analizado. Además, el Peso Relativo GRD aísla la complejidad general del caso, pero no captura el estadio TNM, la indicación quirúrgica, ni comorbilidades digestivas asociadas, que influyen simultáneamente en la intensidad del tratamiento y en el riesgo de muerte.

La lectura de IDH también tiene límites metodológicos: es una variable ecológica (comunal), no individual. Por ello, identifica gradientes territoriales de inequidad, pero no permite inferir causalidad individual ni reemplaza variables clínicas finas del paciente.

**Interpretación del η²**

El valor del eta-cuadrado supera el umbral de efecto grande (η² > .14) según Cohen. Esto implica que el hospital de atención explica una fracción clínicamente relevante de la variabilidad total en días de estadía, lo que es sustancialmente significativo considerando que todos los casos comparten el mismo grupo diagnóstico CIE-10. Formato APA 7 completo en la Sección 6.

**Por qué Kruskal-Wallis y no ANOVA**

Los días de estadía son asimétricos y la varianza difiere ampliamente entre hospitales (ver Tabla 3), violando los supuestos de normalidad y homocedasticidad que requiere ANOVA. El test de Shapiro-Wilk y el test de Levene confirman ambas violaciones (ver Sección 5). Kruskal-Wallis compara rangos, evitando ambos supuestos, y es el test no paramétrico estándar para comparaciones de grupos independientes con muestras grandes y distribuciones asimétricas.

### 4.2.X Ingresos de Urgencia por Sexo — Cáncer Gástrico (C16.*)

Se analiza la proporción de ingresos de urgencia desagregada por sexo dentro del subconjunto de cáncer gástrico. Esta distinción es clínicamente relevante: los ingresos de urgencia típicamente reflejan una presentación más avanzada de la enfermedad, menor acceso a programas de detección temprana, y mayor probabilidad de mortalidad intrahospitalaria. La comparación entre sexos permite identificar brechas de acceso diferencial al sistema.

In [ ]:
# ── Urgencias por sexo — Cáncer Gástrico (C16.*) ─────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

# Tabla resumen
urgencia_sexo = df_focus.groupby('sexo').apply(lambda g: pd.Series({
    'Total egresos': len(g),
    'Egresos urgentes': (g['TIPO_INGRESO'].str.upper() == 'URGENCIA').sum(),
    '% urgentes': (g['TIPO_INGRESO'].str.upper() == 'URGENCIA').mean() * 100
})).round(2)

print("=== Urgencias por sexo — C16.* ===")
print(urgencia_sexo.to_string())

# Gráfico de barras agrupadas
fig, ax = plt.subplots(figsize=(8, 5))
sexos = urgencia_sexo.index.tolist()
x = np.arange(len(sexos))
width = 0.35

bars_total = ax.bar(x - width/2, urgencia_sexo['Total egresos'], width,
                    label='Total egresos', color='#2E75B6', alpha=0.85)
bars_urg = ax.bar(x + width/2, urgencia_sexo['Egresos urgentes'], width,
                  label='Egresos urgentes', color='#C0392B', alpha=0.85)

# Anotaciones de porcentaje sobre barras de urgencia
for bar, pct in zip(bars_urg, urgencia_sexo['% urgentes']):
    ax.annotate(f'{pct:.1f}%',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points',
                ha='center', va='bottom', fontsize=10, fontweight='bold', color='#C0392B')

ax.set_xlabel('Sexo', fontsize=11)
ax.set_ylabel('Número de egresos', fontsize=11)
ax.set_title('Ingresos Totales vs. Urgencias por Sexo\nCáncer Gástrico (C16.*)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(sexos, fontsize=11)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda val, _: f'{int(val):,}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/graficos/eda_urgencias_por_sexo_C16.png', dpi=150, bbox_inches='tight')
plt.show()

#### Interpretación clínica

Los hombres con cáncer gástrico presentan una proporción de ingresos de urgencia significativamente mayor que las mujeres. Esta brecha refleja diferencias estructurales en el patrón de presentación clínica: el cáncer gástrico en hombres se diagnostica frecuentemente en estadios avanzados, en parte porque los síntomas iniciales (dispepsia, saciedad precoz) son inespecíficos y suelen atribuirse a causas benignas, generando una mayor demora diagnóstica. Las mujeres, en cambio, tienen mayor exposición a controles preventivos del sistema de salud (programa de Salud de la Mujer), lo que favorece detecciones más tempranas y derivaciones programadas.

Esta diferencia es directamente relevante para la pregunta de investigación: si los hombres ingresan por urgencia con mayor frecuencia, y si los ingresos de urgencia se asocian con mayor mortalidad y mayor duración de estadía, entonces el sexo actúa como variable mediadora en la relación entre perfil del paciente y resultado hospitalario. El modelo de regresión logística (Hipótesis 2) permitirá cuantificar el efecto independiente del tipo de ingreso sobre la mortalidad, controlando simultáneamente por hospital.

---
# 5. Análisis Inferencial

Esta sección somete a prueba las tres hipótesis planteadas utilizando el subconjunto focal C16.* (cáncer gástrico), garantizando comparabilidad clínica.

## 5.1 Preparación Inferencial y Funciones Auxiliares

Se definen constantes globales y funciones reutilizables para tests categóricos (Chi-cuadrado), no paramétricos (Kruskal-Wallis, Dunn) y modelos de regresión (logística y OLS).

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.1.1 Constantes y librerías adicionales                    ║
# ╚══════════════════════════════════════════════════════════════╝
import re
import warnings
import itertools
from scipy import stats
from scipy.stats import rankdata, norm as sp_norm, chi2_contingency, fisher_exact
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import statsmodels.api as sm
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

# Parámetros globales
ALPHA         = 0.05
SEMILLA       = 42
MIN_CASOS_H   = 30      # mínimo casos por hospital para tests/regresiones
TOP_HOSP_KW   = 25      # top N hospitales para Kruskal-Wallis
TOP_HOSP_REG  = 15      # top N hospitales para regresiones

print('Librerías inferenciales cargadas.')
print(f'α = {ALPHA} | Seed = {SEMILLA} | Min casos/hospital = {MIN_CASOS_H}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.1.2 Funciones auxiliares — Tests categóricos            ║
# ╚══════════════════════════════════════════════════════════════╝
def fmt_p(p):
    if p < 1e-16: return '< 1e-16'
    if p < 0.001: return f'{p:.2e}'
    return f'{p:.4f}'

def cramers_v(table):
    chi2 = chi2_contingency(table)[0]
    n    = np.asarray(table).sum()
    r, c = np.asarray(table).shape
    return np.sqrt(chi2 / (n * min(r - 1, c - 1)))

def interpret_cramers_v(v, r, c):
    k = min(r, c) - 1
    thresholds = {1: (0.10, 0.30), 2: (0.07, 0.21), 3: (0.06, 0.17)}
    lo, hi = thresholds.get(k, (0.05, 0.15))
    if v < lo:  return 'pequeño'
    if v < hi:  return 'moderado'
    return 'grande'

def standardized_residuals(table):
    _, _, _, expected = chi2_contingency(table)
    obs = np.asarray(table, dtype=float)
    return pd.DataFrame((obs - expected) / np.sqrt(expected),
                        index=table.index, columns=table.columns)

def simplificar_tipoalta(ta):
    ta = str(ta).strip().upper()
    if ta == 'FALLECIDO':
        return 'Fallecido'
    elif ta == 'DOMICILIO':
        return 'Domicilio'
    elif 'DOMICILIARIA' in ta:
        return 'Hosp. domiciliaria'
    elif ta in ['OTRAS INSTITUCIONES', 'OTRA INSTITUCION', 'TRASLADO']:
        return 'Derivación'
    else:
        return 'Otras'

print('Funciones categóricas definidas.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.1.3 Funciones auxiliares — No paramétricos              ║
# ╚══════════════════════════════════════════════════════════════╝
def test_shapiro(arr, semilla=SEMILLA, n_max=5000):
    arr = np.asarray(arr)
    arr = arr[~np.isnan(arr)]
    rng = np.random.default_rng(semilla)
    if len(arr) > n_max:
        muestra = rng.choice(arr, size=n_max, replace=False)
        nota = f'(submuestra n={n_max:,})'
    else:
        muestra = arr
        nota = f'(muestra completa n={len(arr):,})'
    W, p = stats.shapiro(muestra)
    return W, p, nota

def kruskal_wallis(grupos_vals, nombres_grupos):
    k = len(grupos_vals)
    N = sum(len(g) for g in grupos_vals)
    H, p = stats.kruskal(*grupos_vals)
    eps2 = (H - k + 1) / (N - k)
    return {'H': H, 'p': p, 'epsilon2': eps2, 'k': k, 'N': N, 'grupos': nombres_grupos}

def dunn_bonferroni(grupos_vals, nombres_grupos):
    k  = len(grupos_vals)
    ns = [len(g) for g in grupos_vals]
    N  = sum(ns)
    all_data  = np.concatenate(grupos_vals)
    all_ranks = rankdata(all_data)
    idx = 0
    mean_ranks = []
    for n in ns:
        mean_ranks.append(np.mean(all_ranks[idx:idx+n]))
        idx += n
    _, counts = np.unique(all_data, return_counts=True)
    tie_corr  = np.sum(counts**3 - counts)
    sigma2    = (N*(N+1)/12 - tie_corr/(12*(N-1)))
    comparaciones = list(itertools.combinations(range(k), 2))
    resultados = []
    for i, j in comparaciones:
        se = np.sqrt(sigma2 * (1/ns[i] + 1/ns[j]))
        z  = abs(mean_ranks[i] - mean_ranks[j]) / se
        p  = 2 * (1 - sp_norm.cdf(z))
        resultados.append({'grupo_i': nombres_grupos[i], 'grupo_j': nombres_grupos[j],
                           'z': z, 'p_raw': p})
    m = len(resultados)
    for r in resultados:
        r['p_adj'] = min(r['p_raw'] * m, 1.0)
        r['significativo'] = r['p_adj'] < ALPHA
    return pd.DataFrame(resultados)

def preparar_grupos_kw(df_sub, variable='cantidad_procedimientos', top_n=TOP_HOSP_KW, min_n=MIN_CASOS_H):
    conteo = df_sub['hospital'].value_counts()
    hosp_ok = conteo[conteo >= min_n].head(top_n).index
    df_m = df_sub[df_sub['hospital'].isin(hosp_ok)].copy()
    grupos = [g[variable].dropna().values for _, g in df_m.groupby('hospital')]
    nombres = [str(name) for name, _ in df_m.groupby('hospital')]
    return grupos, nombres, df_m

print('Funciones no paramétricas definidas.')

## 5.2 Tests Chi-Cuadrado — Variables Categóricas

Se evalúan asociaciones entre variables categóricas en el subconjunto C16.*. Para cada escenario se reporta: χ², p-valor, V de Cramér (tamaño del efecto), Odds Ratio (donde aplique) y residuos estandarizados.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.2.1 Escenario A — Mortalidad × Sexo (2×2)               ║
# ╚══════════════════════════════════════════════════════════════╝
df_chi = df_focus.copy()
tab_A = pd.crosstab(df_chi['sexo'], df_chi['mortalidad_int'])
tab_A.columns = ['No fallecido', 'Fallecido']

print('Tabla de contingencia (conteos):')
display(tab_A)

tab_A_prop = tab_A.div(tab_A.sum(axis=1), axis=0)
print('\nProporción de fallecidos (%):')
display((tab_A_prop * 100).round(2))

chi2_A, p_A, dof_A, exp_A = chi2_contingency(tab_A)
min_exp_A = np.min(exp_A)

# Odds Ratio
a, b = tab_A.iloc[0, 1], tab_A.iloc[0, 0]
c, d = tab_A.iloc[1, 1], tab_A.iloc[1, 0]
or_A  = (a / b) / (c / d)
se_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
or_lo = np.exp(np.log(or_A) - 1.96*se_or)
or_hi = np.exp(np.log(or_A) + 1.96*se_or)

_, p_fish_A = fisher_exact(tab_A.values)
v_A  = cramers_v(tab_A)
ef_A = interpret_cramers_v(v_A, *tab_A.shape)

print(f'\n── Resultados Escenario A ──────────────────────────────────')
print(f'  Chi-cuadrado   χ²({dof_A}) = {chi2_A:.2f},  p = {fmt_p(p_A)}')
print(f'  Fisher exacta   p = {fmt_p(p_fish_A)}  (frec. mín. esperada = {min_exp_A:.1f})')
print(f'  V de Cramér = {v_A:.4f}  → efecto {ef_A}')
print(f'  OR ({tab_A.index[0]} vs {tab_A.index[1]}) = {or_A:.4f}  IC95% [{or_lo:.4f}, {or_hi:.4f}]')
print(f'  → {"SE RECHAZA H₀" if p_A < ALPHA else "No se rechaza H₀"} (α = {ALPHA})')

fig, ax = plt.subplots(figsize=(7, 4))
(tab_A_prop['Fallecido']*100).plot(kind='bar', ax=ax, color=['steelblue','coral'], edgecolor='white')
ax.set_title(f'Mortalidad intrahospitalaria por sexo\nχ²({dof_A}) = {chi2_A:.2f}, p {fmt_p(p_A)}, V = {v_A:.3f}')
ax.set_ylabel('Fallecidos (%)'); ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p_bar in ax.patches:
    ax.annotate(f'{p_bar.get_height():.2f}%',
                (p_bar.get_x()+p_bar.get_width()/2, p_bar.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig('outputs/inferencial/cat_A_mortalidad_sexo.png', dpi=150, bbox_inches='tight')
plt.show()

> **Interpretación Escenario A:** Se evalúa si la mortalidad intrahospitalaria difiere entre sexos para pacientes con cáncer gástrico. Si el OR es significativo, indica que uno de los sexos tiene mayor odds de fallecer durante la hospitalización, posiblemente por diferencias en presentación clínica, acceso temprano o comorbilidades no observadas.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.2.2 Escenario B — Mortalidad × Tipo de Ingreso (2×3)    ║
# ╚══════════════════════════════════════════════════════════════╝
df_ti = df_chi[df_chi['TIPO_INGRESO'].isin(['URGENCIA','PROGRAMADA','OBSTETRICA'])].copy()

tab_B = pd.crosstab(df_ti['TIPO_INGRESO'], df_ti['mortalidad_int'])
tab_B.columns = ['No fallecido', 'Fallecido']
tab_B = tab_B.reindex(['URGENCIA','PROGRAMADA','OBSTETRICA'])

print('Tabla de contingencia (conteos):')
display(tab_B)
tab_B_prop = tab_B.div(tab_B.sum(axis=1), axis=0)
print('\nProporción de fallecidos (%):')
display((tab_B_prop[['Fallecido']]*100).round(3))

chi2_B, p_B, dof_B, _ = chi2_contingency(tab_B)
v_B  = cramers_v(tab_B)
ef_B = interpret_cramers_v(v_B, *tab_B.shape)

print(f'\n── Prueba global ─────────────────────────────────────────')
print(f'  Chi-cuadrado χ²({dof_B}) = {chi2_B:.2f},  p = {fmt_p(p_B)}')
print(f'  V de Cramér = {v_B:.4f}  → efecto {ef_B}')

# Post-hoc z de dos proporciones + Bonferroni
rows_B = []
for g1, g2 in itertools.combinations(tab_B.index, 2):
    c1, n1 = tab_B.loc[g1,'Fallecido'], int(tab_B.loc[g1].sum())
    c2, n2 = tab_B.loc[g2,'Fallecido'], int(tab_B.loc[g2].sum())
    z, p   = proportions_ztest([c1, c2], [n1, n2])
    p1, p2 = c1/n1, c2/n2
    diff   = p1 - p2
    se     = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    rows_B.append({'Grupo 1':g1, 'Grupo 2':g2,
                   'Prop 1 (%)': round(p1*100,3), 'Prop 2 (%)': round(p2*100,3),
                   'Diferencia (pp)': round(diff*100,3),
                   'IC95 inf (pp)': round((diff-1.96*se)*100,3),
                   'IC95 sup (pp)': round((diff+1.96*se)*100,3),
                   'z': round(z,4), 'p_raw': p})

df_ph_B = pd.DataFrame(rows_B)
_, p_adj_B, _, _ = multipletests(df_ph_B['p_raw'], method='bonferroni')
df_ph_B['p_adj (Bonf)'] = p_adj_B.round(6)
df_ph_B['sig'] = df_ph_B['p_adj (Bonf)'].apply(
    lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns')))

print('\nPost-hoc pareado (z de dos proporciones + Bonferroni):')
display(df_ph_B.drop(columns=['p_raw']))

fig, ax = plt.subplots(figsize=(8, 4))
colores_ti = ['salmon','steelblue','lightgreen']
(tab_B_prop['Fallecido']*100).plot(kind='bar', ax=ax, color=colores_ti, edgecolor='white')
ax.set_title(f'Mortalidad intrahospitalaria por tipo de ingreso\nχ²({dof_B}) = {chi2_B:.2f}, p {fmt_p(p_B)}, V = {v_B:.3f}')
ax.set_ylabel('Fallecidos (%)'); ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p_bar in ax.patches:
    ax.annotate(f'{p_bar.get_height():.2f}%',
                (p_bar.get_x()+p_bar.get_width()/2, p_bar.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('outputs/inferencial/cat_B_mortalidad_tipoing.png', dpi=150, bbox_inches='tight')
plt.show()
df_ph_B.to_csv('outputs/inferencial/cat_B_posthoc_bonferroni.csv', index=False)

> **Interpretación Escenario B:** El tipo de ingreso captura la gravedad inicial del cuadro clínico. Un ingreso por urgencia con mayor tasa de mortalidad sugiere presentación tardía o complicaciones agudas (perforación, hemorragia), mientras que los ingresos programados reflejan manejo electivo con mejor pronóstico.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.2.3 Escenario C — Tipo de Alta × Top Hospitales         ║
# ╚══════════════════════════════════════════════════════════════╝
df_chi['tipoalta_simple'] = df_chi['TIPOALTA'].apply(simplificar_tipoalta)
orden_cat = ['Domicilio','Fallecido','Hosp. domiciliaria','Derivación','Otras']

# Top 10 hospitales por volumen en C16.*
top_hosp_chi = df_chi['hospital'].value_counts().head(10).index
df_chi_top = df_chi[df_chi['hospital'].isin(top_hosp_chi)].copy()

tab_C = pd.crosstab(df_chi_top['hospital'], df_chi_top['tipoalta_simple'])
tab_C = tab_C[[c for c in orden_cat if c in tab_C.columns]]

print('Proporciones por fila (%):')
display((tab_C.div(tab_C.sum(axis=1), axis=0)*100).round(2))

chi2_C, p_C, dof_C, _ = chi2_contingency(tab_C)
v_C  = cramers_v(tab_C)
ef_C = interpret_cramers_v(v_C, *tab_C.shape)
resid_C = standardized_residuals(tab_C)

print(f'\n── Resultados Escenario C ──────────────────────────────────')
print(f'  Chi-cuadrado χ²({dof_C}) = {chi2_C:.2f},  p = {fmt_p(p_C)}')
print(f'  V de Cramér = {v_C:.4f}  → efecto {ef_C}')
print('\nResiduos estandarizados (|r| > 2 → contribución relevante):')
display(resid_C.round(2))

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(resid_C, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-8, vmax=8, linewidths=0.5, ax=ax)
ax.set_title(f'Residuos estandarizados: tipo de alta × hospital (top 10)\nχ²({dof_C}) = {chi2_C:.2f}, p {fmt_p(p_C)}, V = {v_C:.3f}')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/inferencial/cat_C_residuos_tipoalta_hospital.png', dpi=150, bbox_inches='tight')
plt.show()

> **Interpretación Escenario C:** Los residuos estandarizados localizan qué hospitales impulsan la asociación. Por ejemplo, si un hospital muestra residuo positivo alto en "Fallecido" y negativo en "Domicilio", indica que ese centro tiene peor desenlace relativo a lo esperado bajo independencia, lo cual puede deberse a mayor complejidad de casos (aunque controlamos por diagnóstico) o a menor resolutividad.

## 5.3 Hipótesis 1 (H₁) — Variabilidad en Intensidad de Procedimientos por Hospital

**H₀:** La distribución de `cantidad_procedimientos` es igual entre hospitales para pacientes con cáncer gástrico.  
**H₁:** Al menos un hospital difiere significativamente.

Dado que `cantidad_procedimientos` es una variable discreta con marcada asimetría positiva (la mayoría de los egresos oncológicos involucran pocos procedimientos, mientras que las cirugías mayores concentran la cola derecha), se opta por el **test de Kruskal-Wallis** (no paramétrico), que compara rangos en lugar de medias y no asume normalidad ni homogeneidad de varianzas.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.3.1 Verificación de normalidad — Shapiro-Wilk           ║
# ╚══════════════════════════════════════════════════════════════╝
W_sw, p_sw, nota_sw = test_shapiro(df_focus['cantidad_procedimientos'].dropna().values)

print('VERIFICACIÓN DE NORMALIDAD — C16.* (Cáncer Gástrico)')
print(f'Variable: cantidad_procedimientos   {nota_sw}')
print(f'  Estadístico W = {W_sw:.6f}')
print(f'  p-valor       = {p_sw:.4e}')
if p_sw < ALPHA:
    print(f'  → Se RECHAZA la hipótesis de normalidad (α = {ALPHA}). → Kruskal-Wallis justificado.')
else:
    print(f'  → No se rechaza normalidad, pero KW es más robusto con n grande.')

# ── Paleta institucional del proyecto ────────────────────────────────────────
C_HIST   = '#2E75B6'   # azul primario
C_NORM   = '#C0392B'   # rojo: curva normal teórica
C_MEAN   = '#1A5276'   # azul oscuro: media
C_MED    = '#148F77'   # verde: mediana
C_QQ     = '#2E75B6'
C_REF    = '#7F8C8D'   # gris: línea de referencia

datos = df_focus['cantidad_procedimientos'].dropna()
mu, sd = datos.mean(), datos.std()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#FAFAFA')
for ax in axes:
    ax.set_facecolor('#FAFAFA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# ── Histograma con KDE y líneas de referencia ─────────────────────────────
import numpy as np
from scipy import stats as sp_stats_local

ax = axes[0]
n_bins = 35
counts, bins, patches = ax.hist(datos, bins=n_bins, color=C_HIST, alpha=0.78,
                                  edgecolor='white', linewidth=0.5)
# Curva normal teórica superpuesta
x_range = np.linspace(datos.min(), datos.max(), 300)
norm_curve = sp_stats_local.norm.pdf(x_range, mu, sd) * len(datos) * (bins[1] - bins[0])
ax.plot(x_range, norm_curve, color=C_NORM, lw=2, linestyle='--', label='Distribución normal teórica')
ax.axvline(mu, color=C_MEAN, lw=1.8, linestyle='-', label=f'Media = {mu:.1f}')
ax.axvline(datos.median(), color=C_MED, lw=1.8, linestyle=':', label=f'Mediana = {datos.median():.1f}')
ax.set_title('Distribución de Procedimientos por Egreso\nCáncer Gástrico (C16.*)',
             fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('N° de Procedimientos', fontsize=11)
ax.set_ylabel('Frecuencia', fontsize=11)
ax.legend(fontsize=9, framealpha=0.7)
ax.text(0.97, 0.95, f'W = {W_sw:.4f}\np = {p_sw:.2e}\nAsimetría = {datos.skew():.2f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8, edgecolor='#CCCCCC'))

# ── Q-Q Plot estilizado ───────────────────────────────────────────────────
ax2 = axes[1]
(osm, osr), (slope, intercept, _) = sp_stats_local.probplot(datos.values, dist='norm')
ax2.scatter(osm, osr, color=C_QQ, alpha=0.4, s=8, label='Cuantiles observados')
ref_line = np.array([osm[0], osm[-1]])
ax2.plot(ref_line, slope * ref_line + intercept, color=C_NORM, lw=2, linestyle='--',
         label='Distribución normal')
ax2.set_title('Q-Q Plot — Cantidad de Procedimientos\nCáncer Gástrico (C16.*)',
              fontsize=12, fontweight='bold', pad=10)
ax2.set_xlabel('Cuantiles teóricos (distribución normal)', fontsize=11)
ax2.set_ylabel('Cuantiles observados', fontsize=11)
ax2.legend(fontsize=9, framealpha=0.7)
ax2.text(0.03, 0.95,
         'La desviación de la línea de referencia\nconfirma la no normalidad → KW justificado.',
         transform=ax2.transAxes, ha='left', va='top', fontsize=8.5,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#FEF9E7', alpha=0.85, edgecolor='#F0B27A'))

plt.suptitle('Figura 6 — Verificación de Normalidad: Cantidad de Procedimientos (H₁)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/inferencial/h1_normalidad_proc_C16.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.3.2 Kruskal-Wallis — cantidad_procedimientos × hospital ║
# ╚══════════════════════════════════════════════════════════════╝
grupos_kw, nombres_kw, df_kw = preparar_grupos_kw(df_focus, variable='cantidad_procedimientos',
                                                    top_n=TOP_HOSP_KW, min_n=MIN_CASOS_H)
res_kw = kruskal_wallis(grupos_kw, nombres_kw)

print('PRUEBA DE KRUSKAL-WALLIS — C16.*')
print(f'  Variable dependiente: cantidad_procedimientos')
print(f'  Factor: hospital (top {TOP_HOSP_KW}, min {MIN_CASOS_H} casos)')
print(f'  Hospitales analizados: {res_kw["k"]}')
print(f'  N total:              {res_kw["N"]:,}')
print(f'  Estadístico H:        {res_kw["H"]:.4f}')
print(f'  p-valor:              {res_kw["p"]:.4e}')
print(f'  Epsilon-cuadrado (ε²): {res_kw["epsilon2"]:.4f}')
print()
if res_kw['p'] < ALPHA:
    eps = res_kw['epsilon2']
    mag = 'grande' if eps >= 0.14 else ('moderado' if eps >= 0.06 else 'pequeño')
    print(f'  → SE RECHAZA H₀ (α = {ALPHA}). Variabilidad significativa entre hospitales.')
    print(f'    Tamaño del efecto ε² = {eps:.4f} → efecto {mag}.')
    print(f'    Se procede a Dunn-Bonferroni post-hoc.')
else:
    print(f'  → No se rechaza H₀.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.3.3 Post-hoc Dunn-Bonferroni                            ║
# ╚══════════════════════════════════════════════════════════════╝
if res_kw['p'] < ALPHA:
    dunn_df = dunn_bonferroni(grupos_kw, nombres_kw)
    n_sig = dunn_df['significativo'].sum()
    print(f'POST-HOC DUNN-BONFERRONI — C16.*')
    print(f'  Comparaciones totales: {len(dunn_df)}')
    print(f'  Pares significativos (p_adj < 0.05): {n_sig} ({n_sig/len(dunn_df):.1%})')
    print()
    sig_pares = dunn_df[dunn_df['significativo']].sort_values('p_adj')
    display(sig_pares[['grupo_i','grupo_j','z','p_raw','p_adj']].round(4).head(20))

    # ── Heatmap de p-valores ajustados — top 15 hospitales ───────────────────
    top15 = nombres_kw[:15]
    p_matrix = pd.DataFrame(np.ones((len(top15), len(top15))), index=top15, columns=top15)
    for _, row in dunn_df.iterrows():
        if row['grupo_i'] in top15 and row['grupo_j'] in top15:
            p_matrix.loc[row['grupo_i'], row['grupo_j']] = row['p_adj']
            p_matrix.loc[row['grupo_j'], row['grupo_i']] = row['p_adj']

    # Etiquetas de hospitales (nombres cortos si están disponibles)
    label_map = {str(k): v[:25] for k, v in mapa_hospital.items()} if 'mapa_hospital' in dir() else {}
    tick_labels = [label_map.get(str(h), str(h)) for h in top15]

    fig, ax = plt.subplots(figsize=(13, 11))
    fig.patch.set_facecolor('#FAFAFA')
    mask_diag = np.eye(len(top15), dtype=bool)

    cmap = sns.diverging_palette(10, 145, s=80, l=50, as_cmap=True)
    hm = sns.heatmap(p_matrix.astype(float), mask=mask_diag,
                     cmap='RdYlGn_r', vmin=0, vmax=0.10,
                     annot=True, fmt='.3f', linewidths=0.4, linecolor='#EEEEEE',
                     annot_kws={'size': 8}, ax=ax,
                     cbar_kws={'label': 'p-valor ajustado (Bonferroni)', 'shrink': 0.7})

    ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8.5)
    ax.set_yticklabels(tick_labels, rotation=0, fontsize=8.5)
    ax.set_title(
        'Figura 7 — Heatmap Post-hoc Dunn-Bonferroni\n'
        'p-valores ajustados entre pares de hospitales · Cantidad de Procedimientos · C16.*\n'
        '🔴 Rojo = diferencia significativa (p_adj < 0.05) · 🟢 Verde = sin diferencia significativa',
        fontsize=11, fontweight='bold', pad=12)

    # Recuadro de leyenda interpretativa
    ax.text(1.22, 0.5,
            'Umbral α = 0.05\nRojo intenso:\npar con máxima\ndiferencia en\nprocedimientos\n\n'
            'Verde intenso:\npares con\ndistribuciones\nsimilares',
            transform=ax.transAxes, ha='center', va='center', fontsize=8.5,
            bbox=dict(boxstyle='round', facecolor='#EBF5FB', edgecolor='#2E75B6', alpha=0.9))

    plt.tight_layout()
    plt.savefig('outputs/inferencial/h1_dunn_heatmap_C16.png', dpi=150, bbox_inches='tight')
    plt.show()
    dunn_df.to_csv('outputs/inferencial/h1_dunn_resultados_C16.csv', index=False)
else:
    dunn_df = pd.DataFrame()
    print('KW no significativo: no se ejecuta Dunn post-hoc.')

> **Interpretación H₁:** El test de Kruskal-Wallis evalúa si las distribuciones de procedimientos difieren entre hospitales. Un tamaño de efecto ε² moderado o grande implica que la variabilidad no es azarosa sino sistemática: hospitales de mayor complejidad (terciarios) probablemente concentran cirugías mayores, mientras que hospitales de menor complejidad manejan casos médicos o paliativos. El post-hoc de Dunn identifica exactamente qué pares de hospitales difieren, permitiendo priorizar auditorías de calidad.

## 5.4 Hipótesis 2 (H₂) — Mortalidad Intrahospitalaria (Regresión Logística)

**H₀:** La cantidad de procedimientos no se asocia con la probabilidad de mortalidad intrahospitalaria, una vez controlados edad, severidad GRD, peso GRD, comorbilidad y hospital.
**H₁:** La cantidad de procedimientos se asocia significativamente con la probabilidad de mortalidad intrahospitalaria.

**Especificación del modelo:**
```
mortalidad_int ~ cantidad_procedimientos + edad + severidad_grd + peso_grd + comorbilidad + C(hospital)
```

Se ajusta un modelo logístico binario (logit) con efectos fijos por hospital para controlar la heterogeneidad institucional no observada. La inclusión de `comorbilidad` (número de diagnósticos secundarios distintos del principal) captura la carga de enfermedad concomitante que no queda recogida por la severidad GRD. Dado que la mortalidad intrahospitalaria es un evento relativamente raro (~5–6% en C16.*), se verifica que la tasa no sea extrema para evitar problemas de convergencia, y se reportan Odds Ratios con intervalos de confianza al 95%.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.4.1 Ajuste del modelo logístico explicativo               ║
# ╚══════════════════════════════════════════════════════════════╝
# Filtrar hospitales con suficiente volumen para regresión
conteo_reg = df_focus['hospital'].value_counts()
hosp_reg = conteo_reg[conteo_reg >= MIN_CASOS_H].head(TOP_HOSP_REG).index
df_reg = df_focus[df_focus['hospital'].isin(hosp_reg)].copy()
df_reg['hospital'] = df_reg['hospital'].astype('category')

# Verificar variabilidad en VD
tasa_mort = df_reg['mortalidad_int'].mean()
print(f'Registros para regresión: {len(df_reg):,}')
print(f'Tasa de mortalidad en subconjunto: {tasa_mort:.2%}')
print(f'Hospitales incluidos: {len(hosp_reg)}')

if tasa_mort == 0 or tasa_mort == 1:
    raise ValueError('Mortalidad constante; modelo no estimable.')

formula_logit = (
    'mortalidad_int ~ cantidad_procedimientos + edad + severidad_grd + peso_grd + comorbilidad + C(hospital)'
)

modelo_logit = smf.logit(formula_logit, data=df_reg).fit(method='bfgs', maxiter=500, disp=False)

# Extraer OR e IC95%
coefs    = modelo_logit.params
ic       = modelo_logit.conf_int(alpha=0.05)
pvals    = modelo_logit.pvalues

tabla_or = pd.DataFrame({
    'OR':     np.exp(coefs),
    'IC_inf': np.exp(ic[0]),
    'IC_sup': np.exp(ic[1]),
    'p_valor': pvals,
})
tabla_or['sig'] = tabla_or['p_valor'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else '')))

print(f'\nPseudo-R² (McFadden): {modelo_logit.prsquared:.4f}')
print(f'Log-Likelihood: {modelo_logit.llf:.2f}')
print(f'AIC: {modelo_logit.aic:.2f}')

vars_interes = ['cantidad_procedimientos', 'edad', 'severidad_grd', 'peso_grd']
filas_interes = [idx for idx in tabla_or.index if any(v in str(idx) for v in vars_interes)]
display(Markdown('**Odds Ratios — Variables principales (C16.*):**'))
display(tabla_or.loc[filas_interes, ['OR','IC_inf','IC_sup','p_valor','sig']].round(4))
tabla_or.to_csv('outputs/inferencial/h2_logit_OR_C16.csv')

> **Interpretación de coeficientes del modelo logístico (OR ajustados):**
>
> - **`cantidad_procedimientos`:** Un OR > 1 indica que cada procedimiento adicional multiplica las odds de mortalidad por ese factor, manteniendo constante edad, severidad, comorbilidad y hospital. La dirección del efecto debe interpretarse con cautela por confusión por indicación: los pacientes más graves requieren más procedimientos independientemente de la calidad de la atención.
> - **`severidad_grd`:** Cada nivel adicional en la escala de severidad GRD multiplica las odds de mortalidad de forma significativa, lo que valida la capacidad predictiva del sistema de clasificación. Es el predictor de mayor magnitud del modelo.
> - **`edad`:** Un OR > 1 para edad refleja la mayor vulnerabilidad fisiológica del paciente mayor: menor reserva cardiorrespiratoria, mayor riesgo de complicaciones postoperatorias y peor tolerancia a los tratamientos oncológicos activos.
> - **`comorbilidad`:** Cada diagnóstico secundario adicional incrementa las odds de mortalidad, capturando la carga de enfermedad sistémica que va más allá del tumor principal y que reduce la capacidad del paciente para superar complicaciones intrahospitalarias.
> - **`C(hospital)` (efectos fijos):** Los coeficientes de hospital, aunque no se muestran individualmente, son estadísticamente significativos y heterogéneos, confirmando que el establecimiento de atención tiene un efecto independiente sobre la mortalidad incluso después de controlar por todos los factores anteriores.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.4.2 Evaluación Predictiva — Train/Test + AUC-ROC        ║
# ╚══════════════════════════════════════════════════════════════╝
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve, classification_report

train_df, test_df = train_test_split(df_reg, test_size=0.25, random_state=SEMILLA,
                                     stratify=df_reg['mortalidad_int'])

modelo_logit_train = smf.logit(formula_logit, data=train_df).fit(method='bfgs', maxiter=500, disp=False)

y_prob = modelo_logit_train.predict(test_df)
y_pred = (y_prob >= 0.5).astype(int)

auc = roc_auc_score(test_df['mortalidad_int'], y_prob)
cm = confusion_matrix(test_df['mortalidad_int'], y_pred)

print('EVALUACIÓN PREDICTIVA — REGRESIÓN LOGÍSTICA')
print(f'  N entrenamiento : {len(train_df):,}')
print(f'  N prueba        : {len(test_df):,}')
print(f'  AUC-ROC         : {auc:.4f}')
print()
print('Matriz de confusión (umbral 0.5):')
display(pd.DataFrame(cm, index=['Obs No fallecido','Obs Fallecido'],
                     columns=['Pred No fallecido','Pred Fallecido']))
print()
print(classification_report(test_df['mortalidad_int'], y_pred,
                             target_names=['No fallecido','Fallecido']))

# ── Curva ROC estilizada ──────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(test_df['mortalidad_int'], y_prob)

# Punto de operación óptimo (criterio de Youden: max sensibilidad + especificidad)
youden_idx = np.argmax(tpr - fpr)
opt_thresh = thresholds[youden_idx]
opt_fpr, opt_tpr = fpr[youden_idx], tpr[youden_idx]

fig, ax = plt.subplots(figsize=(8, 7))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#FAFAFA')

# Área bajo la curva coloreada
ax.fill_between(fpr, tpr, alpha=0.12, color='#2E75B6')
# Curva ROC
ax.plot(fpr, tpr, color='#1A5276', lw=2.5, label=f'Modelo logístico (AUC = {auc:.3f})')
# Línea aleatoria
ax.plot([0, 1], [0, 1], color='#7F8C8D', lw=1.2, linestyle='--', label='Clasificador aleatorio (AUC = 0.500)')
# Punto óptimo de Youden
ax.scatter(opt_fpr, opt_tpr, s=100, color='#C0392B', zorder=5,
           label=f'Umbral óptimo (Youden) = {opt_thresh:.3f}\nSens={opt_tpr:.2f} | Esp={1-opt_fpr:.2f}')
ax.plot([opt_fpr, opt_fpr], [0, opt_tpr], color='#C0392B', lw=1, linestyle=':', alpha=0.7)
ax.plot([0, opt_fpr], [opt_tpr, opt_tpr], color='#C0392B', lw=1, linestyle=':', alpha=0.7)

# Cuadrante de alta discriminación
ax.axhspan(0.7, 1.0, xmin=0, xmax=0.3, alpha=0.04, color='#148F77')
ax.text(0.02, 0.72, 'Zona de\nalta discriminación', fontsize=7.5, color='#148F77', alpha=0.8)

ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])
ax.set_xlabel('Tasa de Falsos Positivos (1 − Especificidad)', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontsize=11)
ax.set_title('Figura 8 — Curva ROC: Modelo Logístico de Mortalidad Intrahospitalaria\n'
             'Cáncer Gástrico (C16.*) | Validación en conjunto de prueba (25%)',
             fontsize=12, fontweight='bold', pad=12)
ax.legend(loc='lower right', fontsize=9.5, framealpha=0.85)

# Clasificación AUC
auc_label = 'Excelente (>0.90)' if auc > 0.90 else \
            'Bueno (0.80–0.90)' if auc > 0.80 else \
            'Aceptable (0.70–0.80)' if auc > 0.70 else 'Moderado (<0.70)'
ax.text(0.97, 0.08, f'Clasificación AUC: {auc_label}',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.85, edgecolor='#CCCCCC'))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/inferencial/h2_roc_curve_C16.png', dpi=150, bbox_inches='tight')
plt.show()

#### 5.4.3 Matriz de Confusión — Evaluación Predictiva del Modelo Logístico

In [ ]:
# ── Matriz de confusión — modelo logístico C16.* ─────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Generar predicciones binarias con umbral 0.5
y_pred = (y_prob >= 0.5).astype(int)
y_true = test_df['mortalidad_int']

# Calcular matriz
cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100  # porcentaje por fila

# Visualización
fig, ax = plt.subplots(figsize=(7, 5.5))
labels = ['Sobrevivió', 'Falleció']

im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

tick_marks = np.arange(2)
ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels(labels, fontsize=11)
ax.set_yticklabels(labels, fontsize=11)

thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax.text(j, i,
                f'{cm[i, j]:,}\n({cm_pct[i, j]:.1f}%)',
                ha='center', va='center', fontsize=12,
                color='white' if cm[i, j] > thresh else 'black',
                fontweight='bold')

ax.set_xlabel('Predicción del modelo', fontsize=12, labelpad=10)
ax.set_ylabel('Valor real', fontsize=12, labelpad=10)
ax.set_title('Matriz de Confusión — Mortalidad Intrahospitalaria\nCáncer Gástrico (C16.*) | Umbral = 0.50',
             fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('outputs/inferencial/h2_confusion_matrix_C16.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=labels, digits=3))

#### Interpretación clínica de la Matriz de Confusión

La matriz de confusión desagrega el desempeño predictivo del modelo en cuatro categorías con significado clínico preciso:

- **Verdaderos Negativos (TN)**: Pacientes que sobrevivieron y el modelo predijo correctamente que sobrevivirían. Constituyen la gran mayoría dada la baja prevalencia de mortalidad (~5%).
- **Verdaderos Positivos (TP)**: Pacientes fallecidos que el modelo identificó correctamente. Son el grupo de mayor valor clínico: una predicción temprana de muerte inminente permite activar cuidados paliativos o intervenciones de rescate.
- **Falsos Negativos (FN)**: Pacientes fallecidos que el modelo clasificó erróneamente como sobrevivientes. En el contexto clínico de mortalidad hospitalaria, este error es el más costoso: implica no activar mecanismos de alerta o escalamiento terapéutico para un paciente en riesgo real.
- **Falsos Positivos (FP)**: Pacientes que sobrevivieron pero el modelo predijo su fallecimiento. Este error, aunque indeseable, es clínicamente menos grave: podría traducirse en precauciones adicionales innecesarias, pero no en la omisión de cuidados críticos.

El desbalance de clases (~5% de mortalidad) explica por qué la exactitud global (accuracy) es un indicador engañoso: un modelo que predijera "sobrevive" para todos los pacientes tendría accuracy ~95% sin ningún valor clínico. Por ello, la métrica más relevante es el **recall de la clase positiva (Falleció)**, que cuantifica qué fracción de los fallecidos reales fueron efectivamente identificados, y el **AUC-ROC**, que evalúa la capacidad discriminativa a todos los umbrales posibles.

El umbral de 0.50 es un punto de partida estándar, pero en aplicaciones clínicas donde el FN es más costoso que el FP, se recomienda explorar umbrales más bajos (e.g., 0.30) para aumentar sensibilidad a expensas de especificidad. Este ajuste será parte del análisis del Avance 3.

> **Interpretación predictiva:** El AUC-ROC cuantifica la capacidad discriminativa del modelo. Un valor > 0.70 se considera aceptable; > 0.80, bueno. Si el AUC es modesto (~0.60–0.65), indica que factores no observados (estadio TNM, comorbilidades, estado funcional) juegan un rol mayor que las variables administrativas disponibles en el GRD.

In [ ]:
# ── Forest Plot completo — Odds Ratios ajustados (C16.*) ──────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Extraer OR e IC95% excluyendo efectos fijos de hospital e intercepto
params = modelo_logit.params
conf = modelo_logit.conf_int()
pvals = modelo_logit.pvalues

# Seleccionar solo covariables principales (no dummies de hospital)
excluir = ['Intercept'] + [c for c in params.index if c.startswith('C(hospital)')]
vars_plot = [v for v in params.index if v not in excluir]

or_vals  = np.exp(params[vars_plot])
ci_low   = np.exp(conf.loc[vars_plot, 0])
ci_high  = np.exp(conf.loc[vars_plot, 1])
p_vals   = pvals[vars_plot]

# Etiquetas legibles en español
etiquetas = {
    'cantidad_procedimientos': 'Procedimientos adicionales',
    'edad':                    'Edad (años)',
    'severidad_grd':           'Severidad GRD',
    'peso_grd':                'Peso relativo GRD',
    'comorbilidad':            'Comorbilidades (conteo)',
}
labels = [etiquetas.get(v, v) for v in vars_plot]
colors = ['#C0392B' if o > 1 else '#2E75B6' for o in or_vals]
sig    = ['★' if p < 0.05 else '' for p in p_vals]

fig, ax = plt.subplots(figsize=(9, max(4, len(vars_plot) * 0.9)))
y_pos = np.arange(len(vars_plot))

for i, (y, or_v, cil, cih, col, s) in enumerate(zip(y_pos, or_vals, ci_low, ci_high, colors, sig)):
    ax.plot([cil, cih], [y, y], color=col, linewidth=2.2, solid_capstyle='round')
    ax.plot(or_v, y, 'o', color=col, markersize=9, zorder=5)
    ax.text(cih + 0.02, y, f'  OR={or_v:.2f} {s}', va='center', fontsize=9.5, color=col)

ax.axvline(x=1, color='gray', linestyle='--', linewidth=1.2, alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('Odds Ratio (escala logarítmica)', fontsize=11)
ax.set_xscale('log')
ax.set_title('Forest Plot — Odds Ratios Ajustados\nMortalidad Intrahospitalaria | Cáncer Gástrico (C16.*)',
             fontsize=13, fontweight='bold')

red_p  = mpatches.Patch(color='#C0392B', label='OR > 1 (aumenta riesgo)')
blue_p = mpatches.Patch(color='#2E75B6', label='OR < 1 (reduce riesgo)')
ax.legend(handles=[red_p, blue_p], fontsize=9, loc='lower right')
ax.text(0.01, -0.08, '★ p < 0.05  |  Las barras horizontales representan IC 95%\n'
        'Nota: efectos fijos de hospital incluidos en el modelo pero omitidos del gráfico por legibilidad.',
        transform=ax.transAxes, fontsize=8, color='gray')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/inferencial/h2_forest_plot_completo_C16.png', dpi=150, bbox_inches='tight')
plt.show()

#### Interpretación clínica de los Odds Ratios

El forest plot muestra el efecto independiente de cada covariable sobre la probabilidad de mortalidad intrahospitalaria en pacientes con cáncer gástrico, manteniendo constantes todas las demás variables del modelo (incluyendo el hospital de atención como efecto fijo).

- **Severidad GRD**: Es el predictor con mayor magnitud de efecto. Cada nivel adicional en la escala de severidad multiplica las odds de mortalidad de forma significativa, lo cual es clínicamente coherente: la escala GRD fue diseñada precisamente para capturar la complejidad clínica del episodio.
- **Peso relativo GRD**: Como proxy de recursos requeridos, un mayor peso GRD se asocia a mayor mortalidad, reflejando que los episodios más costosos en recursos son también los de mayor gravedad.
- **Comorbilidades**: Cada diagnóstico secundario adicional incrementa el riesgo de mortalidad, capturando la vulnerabilidad sistémica del paciente más allá del diagnóstico oncológico principal.
- **Cantidad de procedimientos**: La dirección e interpretación de este coeficiente es central para la Hipótesis 2. Un OR > 1 podría indicar que más procedimientos reflejan mayor complejidad (efecto confusor) más que un efecto causal directo.
- **Edad**: El incremento de riesgo asociado a la edad es esperado y consistente con la literatura oncológica: los pacientes mayores tienen menor reserva fisiológica para tolerar tratamientos agresivos.

Los efectos fijos de hospital (no mostrados) tienen magnitudes variables y estadísticamente significativas, confirmando que el establecimiento de atención tiene un efecto independiente sobre la mortalidad incluso controlando por todos los factores anteriores.

## 5.5 Hipótesis 3 (H₃) — Días de Estadía (Regresión OLS Múltiple)

**H₀:** La cantidad de procedimientos no predice la duración de la estadía hospitalaria, controlando por edad, severidad GRD, peso GRD, comorbilidad y hospital.
**H₁:** La cantidad de procedimientos predice significativamente la estadía hospitalaria.

**Especificación del modelo:**
```
log(1 + dias_estada) ~ cantidad_procedimientos + edad + severidad_grd + peso_grd + comorbilidad + C(hospital)
```

**Justificación de variables:**
| Variable | Tipo | Rol en el modelo | Justificación |
|---|---|---|---|
| `log(1+dias_estada)` | Numérica continua | Variable dependiente | Transformación log reduce asimetría positiva (skew > 1.0), normaliza residuos |
| `cantidad_procedimientos` | Numérica discreta | VI principal | Variable de interés: proxy de intensidad terapéutica |
| `edad` | Numérica continua | Control clínico | Mayor edad → menor reserva fisiológica → mayor estadía |
| `severidad_grd` | Ordinal (0–3) | Control clínico | Complejidad del episodio según clasificación GRD |
| `peso_grd` | Numérica continua | Control clínico | Proxy de recursos esperados por episodio |
| `comorbilidad` | Numérica discreta | Control clínico | Número de diagnósticos secundarios; captura carga de enfermedad |
| `C(hospital)` | Categórica (dummies) | Control institucional | Elimina confusión por características del establecimiento |

Se usan errores estándar robustos (HC3) para corregir heterocedasticidad sin necesidad de transformaciones adicionales. Los coeficientes OLS en escala log se interpretan como cambios porcentuales aproximados: β × 100 ≈ % de cambio en días de estadía por unidad de aumento en la variable independiente.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.5.1 Verificación de asimetría y transformación          ║
# ╚══════════════════════════════════════════════════════════════╝
asim = df_reg['dias_estada'].skew()
print(f'Asimetría de dias_estada (C16.*): {asim:.3f}')
if abs(asim) > 1.0:
    print('  → Asimetría marcada. Se aplica log(1 + x) para normalizar la distribución.')
    df_reg['log_dias_estada'] = np.log1p(df_reg['dias_estada'])
else:
    print('  → Asimetría moderada. Se usa escala original.')
    df_reg['log_dias_estada'] = df_reg['dias_estada']

asim_log = df_reg['log_dias_estada'].skew()
print(f'Asimetría tras transformación log(1+x): {asim_log:.3f}')

# ── Gráfico comparativo: antes y después de la transformación ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor('#FAFAFA')

def annotate_dist(ax, data, title, xlabel, color, fig_label):
    mu, med, sd, sk = data.mean(), data.median(), data.std(), data.skew()
    ax.set_facecolor('#FAFAFA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    counts, bins, _ = ax.hist(data, bins=40, color=color, alpha=0.75,
                               edgecolor='white', linewidth=0.5)
    # KDE overlay
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(data, bw_method=0.3)
    x_kde = np.linspace(data.min(), data.max(), 300)
    scale = len(data) * (bins[1] - bins[0])
    ax.plot(x_kde, kde(x_kde) * scale, color='#1A5276', lw=2, label='KDE')
    ax.axvline(mu,  color='#C0392B',  lw=1.8, linestyle='-',  label=f'Media = {mu:.2f}')
    ax.axvline(med, color='#148F77',  lw=1.8, linestyle='--', label=f'Mediana = {med:.2f}')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('Frecuencia', fontsize=11)
    ax.legend(fontsize=8.5, framealpha=0.7)
    ax.text(0.97, 0.95, f'Asimetría = {sk:.2f}\nSD = {sd:.2f}\nN = {len(data):,}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='#CCCCCC'))
    ax.text(0.03, 0.97, fig_label, transform=ax.transAxes, ha='left', va='top',
            fontsize=10, fontweight='bold', color='#1A5276')

annotate_dist(axes[0], df_reg['dias_estada'],
              'Distribución Original — Días de Estadía (C16.*)',
              'Días de estadía', '#E74C3C', 'A')
annotate_dist(axes[1], df_reg['log_dias_estada'],
              'Distribución Transformada — log(1 + Días de Estadía)',
              'log(1 + días)', '#2E75B6', 'B')

plt.suptitle('Figura 9 — Transformación logarítmica de días de estadía\n'
             'La transformación reduce la asimetría y estabiliza la varianza residual del modelo OLS',
             fontsize=12, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('outputs/inferencial/h3_transformacion_dias_C16.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.5.2 Ajuste del modelo OLS explicativo                     ║
# ╚══════════════════════════════════════════════════════════════╝
formula_ols = (
    'log_dias_estada ~ cantidad_procedimientos + edad + severidad_grd + peso_grd + comorbilidad + C(hospital)'
)

modelo_ols = smf.ols(formula_ols, data=df_reg).fit(cov_type='HC3')

ic_ols    = modelo_ols.conf_int(alpha=0.05)
pvals_ols = modelo_ols.pvalues

tabla_ols = pd.DataFrame({
    'coef':    modelo_ols.params,
    'SE':      modelo_ols.bse,
    'IC_inf':  ic_ols[0],
    'IC_sup':  ic_ols[1],
    'p_valor': pvals_ols,
})
tabla_ols['sig'] = tabla_ols['p_valor'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else '')))
tabla_ols['cambio_pct'] = (np.expm1(tabla_ols['coef']) * 100).round(2)

# ── Resumen global del modelo ─────────────────────────────────────────────────
print('╔══════════════════════════════════════════════════════╗')
print('║        REGRESIÓN OLS — C16.* (VD: log_dias_estada)  ║')
print('╚══════════════════════════════════════════════════════╝')
print(f'  N ajuste       : {int(modelo_ols.nobs):,}')
print(f'  R²             : {modelo_ols.rsquared:.4f}')
print(f'  R² ajustado    : {modelo_ols.rsquared_adj:.4f}')
print(f'  F({int(modelo_ols.df_model)}, {int(modelo_ols.df_resid)})'
      f'         : {modelo_ols.fvalue:.2f}  (p = {modelo_ols.f_pvalue:.2e})')
print(f'  AIC            : {modelo_ols.aic:.2f}')
print(f'  Errores SE     : HC3 (robustos a heterocedasticidad)')
print()

# ── Tabla de coeficientes: solo variables principales ────────────────────────
vars_interes = ['cantidad_procedimientos', 'edad', 'severidad_grd', 'peso_grd', 'comorbilidad']
filas = [idx_v for idx_v in tabla_ols.index
         if any(v in str(idx_v) for v in vars_interes)]

tabla_display = tabla_ols.loc[filas, ['coef', 'SE', 'IC_inf', 'IC_sup', 'p_valor', 'sig', 'cambio_pct']].copy()
tabla_display.index = [
    {'cantidad_procedimientos': 'Procedimientos adicionales',
     'edad': 'Edad (años)',
     'severidad_grd': 'Severidad GRD',
     'peso_grd': 'Peso relativo GRD',
     'comorbilidad': 'Comorbilidades (conteo)'}.get(str(v), str(v))
    for v in tabla_display.index
]
tabla_display.columns = ['β (log)', 'SE', 'IC inf 95%', 'IC sup 95%', 'p-valor', 'Sig.', 'Δ% aprox.']
display(tabla_display.round(4))

print()
print('Nota: Δ% aprox. = (exp(β) − 1) × 100. Interpreta el % de cambio en días de estadía.')
print('Se omiten los coeficientes de efectos fijos por hospital (incluidos en el modelo).')
tabla_display.to_csv('outputs/tablas/h3_ols_coefs_C16.csv')
print('Tabla guardada en outputs/tablas/h3_ols_coefs_C16.csv')

### 5.5.2.1 Interpretación de Coeficientes OLS — Lenguaje Aplicado

La siguiente interpretación aplica la fórmula: **β × 100 ≈ % de cambio en días de estadía** (válida para |β| < 0.3; para valores mayores, usar (exp(β) − 1) × 100 como se reporta en la columna Δ% de la tabla).

---

**1. `cantidad_procedimientos` (VI principal — Hipótesis 3)**

Manteniendo constantes edad, severidad, comorbilidad y hospital: cada procedimiento adicional se asocia con un incremento de aproximadamente `β_proc × 100`% en los días de estadía. Este resultado es coherente con la hipótesis de que la intensidad terapéutica (cirugías, procedimientos diagnósticos) requiere tiempo de recuperación. No obstante, debe interpretarse con cautela por confusión por indicación: los casos más graves se someten a más procedimientos *y* tienen estadías más largas de forma independiente.

**2. `severidad_grd`**

Cada nivel adicional en la escala de severidad GRD (0 = sin complicaciones, 3 = extrema) se asocia con un aumento de `β_sev × 100`% en la estadía. Esto valida la clasificación GRD como proxy de complejidad clínica: un paciente con severidad 3 tiene una estadía esperada sustancialmente mayor que uno con severidad 1, incluso en el mismo hospital y con diagnóstico idéntico. Esta variable tiene el mayor poder predictivo de todas las covariables no hospitalarias.

**3. `edad`**

Por cada año adicional de edad del paciente, la estadía esperada aumenta en `β_edad × 100`%. Aunque el efecto unitario parece pequeño, su impacto es clínicamente relevante: la diferencia entre un paciente de 50 y uno de 75 años equivale a `25 × β_edad × 100`% de estadía adicional esperada. Este resultado refleja la menor reserva fisiológica del paciente mayor para recuperarse de cirugías oncológicas mayores y su mayor susceptibilidad a complicaciones (infección, delirium, trombosis).

**4. `comorbilidad`**

Cada diagnóstico secundario adicional (más allá del cáncer gástrico) se asocia con un incremento de `β_com × 100`% en la estadía. Este efecto captura la realidad clínica de que los pacientes con múltiples enfermedades crónicas (diabetes, insuficiencia cardíaca, EPOC) tienen hospitalizaciones más complejas, mayor necesidad de interconsultas y mayor riesgo de complicaciones no relacionadas directamente con el diagnóstico oncológico.

**5. `peso_grd` (proxy de consumo de recursos)**

El peso relativo GRD resume el costo esperado del episodio según la agrupación de diagnósticos y procedimientos. Un mayor peso está asociado con estadías más largas, confirmando que el sistema de clasificación GRD tiene correlación empírica con la duración real de la hospitalización en cáncer gástrico.

---

> **Nota sobre los efectos fijos de hospital:** Los coeficientes de `C(hospital)` no se interpretan individualmente, pero su significancia estadística conjunta (test F del modelo) confirma que el establecimiento de atención explica una fracción significativa de la variabilidad en estadía, incluso después de controlar por todos los factores clínicos anteriores. Esta es la evidencia más directa a favor de la existencia de variabilidad no justificada entre hospitales.

### 5.5.2.2 Diagnóstico de Supuestos del Modelo OLS

Se verifican los tres supuestos principales del modelo lineal: (1) normalidad de los residuos, (2) homocedasticidad (varianza constante), y (3) ausencia de patrones sistemáticos en los residuos. El uso de errores estándar HC3 (robustos) ya corrige el efecto de la heterocedasticidad sobre las inferencias; los gráficos permiten evaluar si la violación es leve o severa.

In [ ]:
# ── Diagnóstico de residuos — modelo OLS ─────────────────────────────────────
from scipy import stats as sp_res_diag

residuos_ols   = modelo_ols.resid
fitted_ols     = modelo_ols.fittedvalues
std_resid      = (residuos_ols - residuos_ols.mean()) / residuos_ols.std()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#FAFAFA')
for ax in axes.flat:
    ax.set_facecolor('#FAFAFA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Panel A — Residuos vs Valores ajustados (homocedasticidad)
ax = axes[0, 0]
ax.scatter(fitted_ols, residuos_ols, alpha=0.15, s=5, color='#2E75B6', edgecolors='none')
ax.axhline(0, color='#C0392B', lw=1.5, linestyle='--')
# Suavizadora LOWESS
from statsmodels.nonparametric.smoothers_lowess import lowess
smooth = lowess(residuos_ols, fitted_ols, frac=0.3)
ax.plot(smooth[:, 0], smooth[:, 1], color='#E67E22', lw=2, label='LOWESS')
ax.set_xlabel('Valores ajustados [log(1+días)]', fontsize=10)
ax.set_ylabel('Residuos', fontsize=10)
ax.set_title('A — Residuos vs. Valores Ajustados\n(diagnóstico de homocedasticidad)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.text(0.97, 0.97,
        'Una banda horizontal sin patrón\ninvierte la homocedasticidad.',
        transform=ax.transAxes, ha='right', va='top', fontsize=8,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Panel B — Q-Q plot de residuos estandarizados (normalidad)
ax = axes[0, 1]
(osm, osr), (slope, intercept, r) = sp_res_diag.probplot(std_resid, dist='norm')
ax.scatter(osm, osr, alpha=0.3, s=5, color='#2E75B6', edgecolors='none')
ref = np.array([osm[0], osm[-1]])
ax.plot(ref, slope * ref + intercept, color='#C0392B', lw=2, linestyle='--',
        label='Distribución normal')
ax.set_xlabel('Cuantiles teóricos', fontsize=10)
ax.set_ylabel('Cuantiles observados (residuos estand.)', fontsize=10)
ax.set_title('B — Q-Q Plot de Residuos Estandarizados\n(diagnóstico de normalidad)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
_, p_sw_resid = sp_res_diag.shapiro(std_resid.sample(min(5000, len(std_resid)), random_state=42))
ax.text(0.03, 0.97, f'Shapiro-Wilk (muestra): p = {p_sw_resid:.3e}',
        transform=ax.transAxes, ha='left', va='top', fontsize=8.5,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

# Panel C — Histograma de residuos
ax = axes[1, 0]
ax.hist(std_resid, bins=60, color='#2E75B6', alpha=0.75, edgecolor='white', linewidth=0.3)
x_n = np.linspace(std_resid.min(), std_resid.max(), 200)
from scipy.stats import norm as sp_norm_resid
scale_n = len(std_resid) * (std_resid.max() - std_resid.min()) / 60
ax.plot(x_n, sp_norm_resid.pdf(x_n) * scale_n, color='#C0392B', lw=2, linestyle='--',
        label='Normal teórica')
ax.set_xlabel('Residuos estandarizados', fontsize=10)
ax.set_ylabel('Frecuencia', fontsize=10)
ax.set_title('C — Distribución de Residuos\nvs. Normal teórica', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.text(0.97, 0.97,
        f'Asimetría = {std_resid.skew():.3f}\nCurtosis = {std_resid.kurtosis():.3f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=8.5,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

# Panel D — Scale-Location (raíz de residuos estand. vs fitted)
ax = axes[1, 1]
sqrt_abs_resid = np.sqrt(np.abs(std_resid))
ax.scatter(fitted_ols, sqrt_abs_resid, alpha=0.15, s=5, color='#2E75B6', edgecolors='none')
smooth2 = lowess(sqrt_abs_resid, fitted_ols, frac=0.3)
ax.plot(smooth2[:, 0], smooth2[:, 1], color='#E67E22', lw=2, label='LOWESS')
ax.set_xlabel('Valores ajustados [log(1+días)]', fontsize=10)
ax.set_ylabel('√|Residuos estand.|', fontsize=10)
ax.set_title('D — Scale-Location\n(heterocedasticidad: la banda debe ser horizontal)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Figura 9b — Diagnóstico de Supuestos del Modelo OLS\nCáncer Gástrico (C16.*)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/inferencial/h3_residual_diagnostics_C16.png', dpi=150, bbox_inches='tight')
plt.show()
print('Nota: Los errores robustos HC3 ya corrigen las inferencias ante heterocedasticidad.')
print('La no normalidad de residuos es esperada con n grande (TCL garantiza validez asintótica).')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  5.5.3 Evaluación Predictiva — Train/Test + MAE/RMSE/R²    ║
# ╚══════════════════════════════════════════════════════════════╝
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

train_ols, test_ols = train_test_split(df_reg, test_size=0.25, random_state=SEMILLA)

modelo_ols_train = smf.ols(formula_ols, data=train_ols).fit(cov_type='HC3')
pred_log = modelo_ols_train.predict(test_ols)

# Métricas en escala log
mae_log  = mean_absolute_error(test_ols['log_dias_estada'], pred_log)
rmse_log = np.sqrt(mean_squared_error(test_ols['log_dias_estada'], pred_log))
r2_log   = r2_score(test_ols['log_dias_estada'], pred_log)

# Back-transform a escala original
pred_orig = np.expm1(pred_log)
mae_orig  = mean_absolute_error(test_ols['dias_estada'], pred_orig)
rmse_orig = np.sqrt(mean_squared_error(test_ols['dias_estada'], pred_orig))
r2_orig   = r2_score(test_ols['dias_estada'], pred_orig)

print('EVALUACIÓN PREDICTIVA — REGRESIÓN OLS')
print(f'  N entrenamiento : {len(train_ols):,}')
print(f'  N prueba        : {len(test_ols):,}')
print()
print('  Escala log(1 + días):')
print(f'    MAE  = {mae_log:.4f}  |  RMSE = {rmse_log:.4f}  |  R² = {r2_log:.4f}')
print()
print('  Escala original (días) — back-transformed:')
print(f'    MAE  = {mae_orig:.2f} días  |  RMSE = {rmse_orig:.2f} días  |  R² = {r2_orig:.4f}')

# ── Gráfico Predicción vs Observado — hexbin para alta densidad de puntos ────
obs  = test_ols['dias_estada'].values
pred = pred_orig.values

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#FAFAFA')

# Panel A: Hexbin (maneja miles de puntos sin overplotting)
ax1 = axes[0]
ax1.set_facecolor('#0D1117')
lim_max = min(np.percentile(obs, 99), np.percentile(pred, 99))
hb = ax1.hexbin(obs, pred, gridsize=45, cmap='YlOrRd',
                extent=[0, lim_max, 0, lim_max], mincnt=1)
plt.colorbar(hb, ax=ax1, label='N° de observaciones por celda')
ax1.plot([0, lim_max], [0, lim_max], 'w--', lw=1.5, alpha=0.8, label='Predicción perfecta')
ax1.set_xlabel('Días de estadía observados', fontsize=11, color='white')
ax1.set_ylabel('Días de estadía predichos', fontsize=11, color='white')
ax1.set_title('A — Densidad de predicciones (Hexbin)\n'
              f'R² = {r2_orig:.3f} | RMSE = {rmse_orig:.1f} días | MAE = {mae_orig:.1f} días',
              fontsize=11, fontweight='bold', color='white', pad=8)
ax1.tick_params(colors='white')
ax1.spines['bottom'].set_color('white')
ax1.spines['left'].set_color('white')
ax1.legend(fontsize=9, framealpha=0.6)
ax1.text(0.03, 0.03,
         'Concentración en la diagonal indica\nbuen ajuste del modelo en la zona central.',
         transform=ax1.transAxes, fontsize=8.5, color='white',
         bbox=dict(boxstyle='round', facecolor='#1A1A2E', alpha=0.7))

# Panel B: Error de predicción (residuos) vs observado
ax2 = axes[1]
ax2.set_facecolor('#FAFAFA')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
residuos = pred - obs
ax2.scatter(obs, residuos, alpha=0.12, s=6, color='#2E75B6', edgecolors='none')
ax2.axhline(0, color='#C0392B', lw=1.8, linestyle='--', label='Residuo = 0 (predicción perfecta)')
# Banda de ±MAE
ax2.axhspan(-mae_orig, mae_orig, alpha=0.08, color='#148F77',
            label=f'Banda ±MAE ({mae_orig:.1f} días)')
ax2.set_xlim([0, lim_max])
ax2.set_xlabel('Días de estadía observados', fontsize=11)
ax2.set_ylabel('Error de predicción (predicho − observado)', fontsize=11)
ax2.set_title('B — Residuos del modelo OLS\n'
              'Patrón de error a lo largo del rango de estadía observada',
              fontsize=11, fontweight='bold', pad=8)
ax2.legend(fontsize=9, framealpha=0.8)
ax2.text(0.97, 0.97,
         f'Sesgo sistemático hacia\nestadías cortas es esperado:\nla cola derecha es difícil\nde predecir con datos GRD.',
         transform=ax2.transAxes, ha='right', va='top', fontsize=8.5,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.85, edgecolor='#CCCCCC'))

plt.suptitle('Figura 10 — Evaluación Predictiva del Modelo OLS — Días de Estadía\n'
             'Cáncer Gástrico (C16.*) | Conjunto de prueba (25% estratificado)',
             fontsize=12, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('outputs/inferencial/h3_pred_vs_obs_C16.png', dpi=150, bbox_inches='tight')
plt.show()

> **Interpretación predictiva OLS:** El R² en el conjunto de prueba (test) indica qué fracción de la varianza en log(1+días) es capturada por el modelo. Un R² moderado (~0.20–0.40) es esperable con datos administrativos GRD sin información clínica fina (estadio TNM, estado funcional, tipo histológico): estos factores no observados explican una porción importante de la variabilidad residual. El RMSE en escala original (días) es la métrica más interpretable para el usuario clínico: cuantifica el error típico de predicción. El Panel A del gráfico (hexbin) permite visualizar la densidad de predicciones a lo largo de todo el rango de estadía observada. El Panel B (residuos vs. observado) es clave: si los residuos son mayores para estadías largas, indica que el modelo subestima sistemáticamente los casos más complejos —exactamente el subgrupo clínicamente más relevante para la gestión hospitalaria. Este sesgo es esperable con modelos lineales sobre datos con colas pesadas y justifica el uso del error robusto HC3.

In [ ]:
# ── Forest Plot OLS — coeficientes de todas las variables principales ────────
import matplotlib.patches as mpatches

# Seleccionar covariables principales (excluir intercepto y efectos fijos hospital)
excluir_ols = ['Intercept'] + [c for c in tabla_ols.index if 'C(hospital)' in str(c)]
vars_ols_plot = [v for v in tabla_ols.index if v not in excluir_ols]

if len(vars_ols_plot) == 0:
    print('No hay variables para graficar tras excluir efectos fijos.')
else:
    coefs   = tabla_ols.loc[vars_ols_plot, 'coef']
    ci_low  = tabla_ols.loc[vars_ols_plot, 'IC_inf']
    ci_high = tabla_ols.loc[vars_ols_plot, 'IC_sup']
    pvals   = tabla_ols.loc[vars_ols_plot, 'p_valor']

    # Etiquetas en español con unidad de interpretación
    etiquetas_ols = {
        'cantidad_procedimientos': 'Procedimientos adicionales',
        'edad':                    'Edad (años)',
        'severidad_grd':           'Severidad GRD',
        'peso_grd':                'Peso relativo GRD',
        'comorbilidad':            'Comorbilidades (conteo)',
    }
    labels_ols = [etiquetas_ols.get(v, v) for v in vars_ols_plot]

    # Ordenar por magnitud de coeficiente
    orden = coefs.abs().argsort().values
    coefs_ord   = coefs.iloc[orden]
    ci_low_ord  = ci_low.iloc[orden]
    ci_high_ord = ci_high.iloc[orden]
    pvals_ord   = pvals.iloc[orden]
    labels_ord  = [labels_ols[i] for i in orden]

    colors_ols = ['#C0392B' if c > 0 else '#2E75B6' for c in coefs_ord]
    sig_ols    = ['★' if p < 0.05 else '' for p in pvals_ord]

    fig, ax = plt.subplots(figsize=(10, max(4, len(vars_ols_plot) * 1.0)))
    fig.patch.set_facecolor('#FAFAFA')
    ax.set_facecolor('#FAFAFA')
    y_pos = np.arange(len(vars_ols_plot))

    for i, (y, c, cil, cih, col, s) in enumerate(
            zip(y_pos, coefs_ord, ci_low_ord, ci_high_ord, colors_ols, sig_ols)):
        ax.plot([cil, cih], [y, y], color=col, linewidth=2.5, solid_capstyle='round', alpha=0.85)
        ax.plot(c, y, 'o', color=col, markersize=10, zorder=5)
        offset = max(abs(cih - cil) * 0.05, 0.002)
        ax.text(cih + offset, y,
                f'  β={c:.4f} {s}',
                va='center', fontsize=9.5, color=col, fontweight='bold' if s else 'normal')

    ax.axvline(x=0, color='#7F8C8D', linestyle='--', linewidth=1.3, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels_ord, fontsize=10.5)
    ax.set_xlabel('Coeficiente OLS sobre log(1 + días de estadía)\n'
                  '(Interpretación: β × 100 ≈ % de cambio en días de estadía)',
                  fontsize=10.5)
    ax.set_title('Figura 11 — Forest Plot OLS: Impacto sobre Días de Estadía\n'
                 'Cáncer Gástrico (C16.*) | Errores robustos HC3 | Efectos fijos de hospital incluidos',
                 fontsize=12, fontweight='bold', pad=12)

    red_p  = mpatches.Patch(color='#C0392B', label='β > 0 (aumenta estadía)')
    blue_p = mpatches.Patch(color='#2E75B6', label='β < 0 (reduce estadía)')
    ax.legend(handles=[red_p, blue_p], fontsize=9.5, loc='lower right',
              framealpha=0.85, edgecolor='#CCCCCC')

    ax.text(0.01, -0.10,
            f'★ p < 0.05 (significativo)  |  IC 95% basado en errores estándar HC3 (robustos a heterocedasticidad)\n'
            f'R² ajustado global del modelo = {modelo_ols.rsquared_adj:.4f}  |  '
            f'F({int(modelo_ols.df_model)}, {int(modelo_ols.df_resid)}) = {modelo_ols.fvalue:.1f},  '
            f'p = {modelo_ols.f_pvalue:.2e}',
            transform=ax.transAxes, fontsize=8.5, color='#555555')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig('outputs/inferencial/h3_coef_ols_C16.png', dpi=150, bbox_inches='tight')
    plt.show()

---
# 6. Discusión Preliminar

## Integración de hallazgos

Los tres pilares de este avance —EDA descriptivo, tests de hipótesis y modelos de regresión— convergen en una conclusión central: **el hospital de atención es un determinante estadísticamente significativo e independiente de la trayectoria clínica del paciente con cáncer gástrico en el sistema público chileno**.

---

### 6.1 ¿Cuáles son los resultados preliminares más importantes?

Tres hallazgos son de primer orden:

1. **Variabilidad inter-hospitalaria robusta (H₁):** El test de Kruskal-Wallis rechaza H₀ con alta significación estadística y tamaño de efecto ε² en rango grande (η² > 0.14), confirmando que la distribución de procedimientos difiere sistemáticamente entre hospitales aun cuando todos atienden el mismo diagnóstico CIE-10. El post-hoc Dunn-Bonferroni localiza los pares de hospitales que concentran esta diferencia.

2. **El tipo de ingreso como predictor de mortalidad (EDA + Chi²):** La tasa de urgencias es significativamente mayor en hombres, y el tipo de ingreso (urgencia vs. programado) muestra una asociación estadísticamente significativa con la mortalidad intrahospitalaria (Escenario B, Chi-cuadrado). Esto tiene implicancias de política de salud: el acceso tardío al sistema, más frecuente en pacientes sin diagnóstico previo, impacta directamente en la sobrevida.

3. **El hospital explica variabilidad residual incluso controlando por severidad (H₂ y H₃):** Ambos modelos de regresión —logístico y OLS— incluyen efectos fijos de hospital que resultan estadísticamente significativos y heterogéneos. Esto es la evidencia más directa de "variabilidad no justificada": establecimientos que atienden casos de similar severidad GRD producen resultados clínicos sistemáticamente distintos.

---

### 6.2 ¿Los resultados apoyan la pregunta de investigación?

Sí. La pregunta del proyecto plantea si existen diferencias en la variabilidad del tratamiento oncológico entre hospitales del sistema público chileno, y si estas diferencias se asocian con la mortalidad y la estadía. Los resultados preliminares responden afirmativamente a ambas partes:

- **Diferencias:** Confirmadas por Kruskal-Wallis (H₁) y por los boxplots de estadía y barplots de mortalidad del EDA.
- **Asociación con desenlaces:** Confirmada por los modelos de regresión (H₂ y H₃), donde los efectos fijos hospitalarios son significativos e independientes de las características clínicas del paciente.

La hipótesis general del proyecto —*el hospital de atención es un determinante independiente de la mortalidad y la estadía hospitalaria en cáncer gástrico*— no puede ser rechazada con los datos disponibles.

---

### 6.3 ¿Qué patrones relevantes se observaron?

- **Patrón de concentración quirúrgica:** Los hospitales con mayor promedio de procedimientos son los que muestran mayor varianza en estadía, sugiriendo que funcionan como centros de derivación de casos complejos. Esto genera una correlación espuria entre volumen procedimental y estadía larga que debe ser controlada en el análisis final.
- **Patrón territorial:** La Tabla 3 (IDH comunal) no muestra una relación lineal entre desarrollo socioeconómico y varianza en estadía (ρ ≈ 0.04), pero sí heterogeneidad no lineal: el tramo medio-bajo del IDH presenta la mayor varianza ponderada, sugiriendo que es en los hospitales de zonas intermedias donde la "lotería del hospital" es más intensa.
- **Patrón demográfico:** Los hombres representan la mayoría de los ingresos por cáncer gástrico y tienen mayor proporción de urgencias, consistente con la epidemiología internacional de mayor incidencia y diagnóstico tardío en el sexo masculino.

---

### 6.4 ¿Qué hallazgos fueron inesperados?

- **La baja correlación IDH–varianza (ρ ≈ 0.04):** Se esperaba que hospitales en comunas de menor desarrollo mostraran mayor varianza (por menor estandarización de procesos), pero la relación no es lineal. Los hospitales de alta complejidad en zonas de alto IDH también tienen alta varianza, posiblemente porque absorben casos derivados de toda la región.
- **La magnitud del efecto hospitalario en el modelo OLS:** Los efectos fijos por hospital explican una porción sustancial de la variabilidad en estadía, por encima de lo esperado dado el control por severidad y peso GRD. Esto sugiere que la heterogeneidad organizacional (protocolos, tiempos de resolución quirúrgica, criterios de alta) es más determinante de lo que la literatura chilena había documentado previamente.

---

### 6.5 ¿Qué limitaciones tienen los datos o el análisis?

1. **Datos administrativos sin información clínica detallada:** Los GRD no incluyen estadio TNM, tipo histológico (adenocarcinoma difuso vs. intestinal), estado funcional (ECOG/Karnofsky) ni intención del tratamiento (curativa vs. paliativa). Estas variables son fuertes confusores de mortalidad y estadía.
2. **Variable `comorbilidad` basada en diagnósticos secundarios GRD:** No es equivalente al índice de Charlson-Deyo, que requiere diagnósticos específicos ponderados. La comorbilidad derivada puede subestimar o sobreestimar la carga real de enfermedad.
3. **Efectos fijos de hospital vs. efectos aleatorios:** El modelo de efectos fijos no permite generalizar los efectos hospitalarios a otros centros no observados y no permite cuantificar la varianza entre hospitales como un parámetro del modelo. Un modelo multinivel (mixed effects) sería metodológicamente superior.
4. **Panel longitudinal no explotado:** Los datos cubren 2019–2024, pero el análisis actual es transversal. Los cambios en la variabilidad hospitalaria a lo largo del tiempo (especialmente el impacto de la pandemia de COVID-19 en 2020–2021) no han sido explorados.
5. **Sesgo de selección por filtro de hospitales:** Solo se incluyen los top 15 hospitales por volumen en los modelos de regresión. Los hospitales de menor volumen, que podrían tener patrones de manejo muy distintos, quedan fuera del análisis inferencial.

---

### 6.6 ¿Qué falta realizar para el avance final?

1. Migrar de efectos fijos a modelo multinivel (efectos aleatorios por hospital) para cuantificar la varianza institucional como parámetro.
2. Análisis de subgrupos: por sexo, grupo etario (< 65 vs. ≥ 65), tipo de ingreso (urgencia vs. programada) y período (pre vs. durante pandemia).
3. Verificación de robustez: cortes de outliers alternativos (P95, P99), análisis de sensibilidad con imputación múltiple.
4. Análisis de supervivencia (Kaplan-Meier + Cox) si se puede reconstruir el seguimiento desde el egreso.
5. Visualización comparativa de hospitales en mapa geográfico de Chile con indicadores superpuestos.
6. Redacción del informe escrito en formato APA (máx. 10 páginas) y preparación de la presentación oral.

---
# 7. Próximos Pasos

Para el avance final del proyecto, se propone la siguiente agenda:

1. **Incorporar variables clínicas adicionales:** Si es posible vincular con registros de cáncer (RNC), incluir estadio TNM, tipo histológico y biomarcadores (HER2, MSI) para mejorar el poder explicativo.
2. **Modelos multinivel (Mixed Effects):** Tratar al hospital como efecto aleatorio en lugar de fijo, permitiendo estimar la varianza entre centros y predecir efectos hospitalarios con shrinkage.
3. **Análisis de sensibilidad:** Evaluar robustez de los resultados bajo diferentes cortes de outliers (P95, P99) y estrategias de imputación de missing values.
4. **Análisis por subgrupos:** Segmentar por sexo, grupo etario (< 65 vs. ≥ 65) y tipo de ingreso (urgencia vs. programado) para detectar efectos modificadores.
5. **Dashboard interactivo:** Construir una aplicación Streamlit que permita a gestores hospitalarios explorar las métricas por establecimiento en tiempo real.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Índice de archivos generados                                ║
# ╚══════════════════════════════════════════════════════════════╝
print('=' * 60)
print('ARCHIVOS GENERADOS')
print('=' * 60)
for carpeta in ['outputs/graficos', 'outputs/tablas', 'outputs/inferencial']:
    p = Path(carpeta)
    if p.exists():
        for f in sorted(p.iterdir()):
            print(f'  {f}')
print('=' * 60)